# C13 · Code walkthrough: `examples/halo_sizing.py` (the driver of the one coupled problem)

**Track:** code walkthroughs. Every listing is printed from the repository with its real line numbers
(`show_source`). This file is the driver: it holds the Halo's requirements and assumptions, builds the aircraft from
a `HaloDesign` whose fields are `asb.Opti` variables, writes every constraint of the sizing problem in one function
(`solve_halo_sizing_once`), and wraps that function in the starting-point strategy that makes it converge. All 1,955
lines are shown below, in order.

| Lines | Block | What it holds | Depth here |
|---|---|---|---|
| 1-93 | header | docstring, imports, globals (`fuel_factor`, SOC levels, cost-objective scale) | full |
| 95-128 | equipment | `EquipmentItem`, XV-15 groups, uncrewed adjustments | full |
| 131-169 | `HaloRequirements` | the requirement set, field by field | full |
| 172-424 | `HaloAssumptions` | 143 assumptions and switches, grouped by tier | full |
| 427-455 | `HaloDesign` | every sized quantity (variables or numbers) | full |
| 458-633 | builders | battery, bus window, electrical layer, redundancy, aerodynamics, gear stages | brief (C03, C06, C07, C08) |
| 635-898 | `build_halo_aircraft` | the aircraft from a design | brief (C09) |
| 901-1004 | mission, result, helpers | `halo_mission`, `HaloSizingResult`, start and unit helpers | full |
| 1006-1072 | `solve_halo_sizing` | integer units, staircase fallback, multistart dispatch, strings-off fallback | full |
| 1075-1399 | `solve_halo_sizing_once` | **the problem**: variables, points, closure, margins, cost, objective, extraction | full, plus an x-ray of the solve |
| 1402-1574 | failures and summaries | failure hovers, hover/electrical/thermal/battery summaries | full |
| 1577-1790 | multistart | start labels, `default_starts`, precursors, perturbation, cache | full |
| 1793-1955 | baselines, enumeration | `pre_plan035`, `pre_layout`, named sets, max payload, bus voltage, `__main__` | full |

**Who imports it.** 13 integration tests (`tests/integration/test_halo_*.py`), 3 trajectory tests, 3 aerodynamics
tests, 7 examples (`halo_design_space.py`, `halo_mission_profile.py`, `halo_oew_uncertainty.py`,
`halo_conversion_corridor.py`, `halo_trajectory_computed_corridor.py`, `halo_structure_reference.py`,
`trajectory_optimization.py`) and `tutorials/tutorial_helpers.py` (`baseline()`, `baseline_assumptions()`).

**What it imports.** Almost every package module (L44-83), plus three examples: `xv15_hot_day` (temperature lapse),
`xv15_performance` (the fitted lapse exponent) and `xv15_reference` (the calibration factors). The design rule
(`docs/OPTIMIZATION_PHILOSOPHY.md`, decision 010) is that the major sizing constraints live in this one readable
file, not inside orchestration classes.

**The x-ray.** Section 10.2 re-solves the final problem once, warm-started from the cached baseline, with
`capture_opti()` recording every `subject_to` and its Lagrange multipliers. The result (and one finite-difference
re-solve) is cached with `cached()`, so a re-run of this notebook takes about a minute.

In [1]:
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from tutorial_helpers import *          # paths, plot style, check(), lb(), baseline(), capture_opti()

import casadi as cas
from dataclasses import fields, replace
import examples.halo_sizing as halo
from examples.halo_sizing import *
from examples.halo_sizing import _precursor_problem
from aircraft_closure.powertrain.components.battery_ecm import EquivalentCircuitBattery, inr21700_50g_cell

ref, a, req = baseline(), baseline_assumptions(), HaloRequirements()
d = ref.design
print(f"baseline v3.6: MTOM {ref.mass_takeoff_kg:.2f} kg ({lb(ref.mass_takeoff_kg):,.0f} lb), empty {ref.mass_empty_kg:.1f} kg, "
      f"fuel {ref.mass_fuel_kg:.1f} kg, battery {ref.energy_battery_kWh:.2f} kWh")
print("unit counts (motor, generator):", a.count_units_motor, a.count_units_generator)

baseline v3.6: MTOM 6885.06 kg (15,179 lb), empty 5048.6 kg, fuel 936.4 kg, battery 69.99 kWh
unit counts (motor, generator): 2 3


---
## 1. Header: docstring, imports, globals

In [2]:
show_source("examples/halo_sizing.py", 1, 94)

# examples/halo_sizing.py  lines 1-94
   1  """Halo-class two-rotor series-hybrid tiltrotor: simultaneous sizing, mission and energy allocation.
   2  
   3  The Tier 9 formulation, re-baselined on the Bell XV-15 (plans 011-013):
   4  two tip-mounted rotors, two turbogenerators on one bus with a battery, and the
   5  validated Tier 10 models.
   6  
   7  Requirements (adopted 2026-10-02, revised 2026-10-03): 900 kg payload (780 kg in plan 022, back to
   8  900 kg with the AFDD tiltrotor wing in plan 026), 445 nm mission, 210 kt at 10,000 ft, 13,000 ft ceiling, OGE hover at 4,000 ft, engine-out
   9  hover on one turbogenerator plus battery, stall at most 120 kt.
  10  
  11  Models carried from Tier 10:
  12    * weights: Raymer GA airframe x XV-15 calibration factors; AFDD82 rotor group
  13      and AFDD83 gearboxes; AFDD82 engine section; AeroSandbox turboshaft
  14      mass-power regression as an explicit Opti equality;
  15    * engine: lapse sigma^0.797 and part-power knockd

**Line by line**

- **L1-34 module docstring.** A history of the formulation, tier by tier. It is accurate as a list of what was added
  (Tier 9 formulation re-baselined on the XV-15; Tier 16 hot day; Tier 20 AFDD wing; Tier 19 thermal; Tier 18
  redundancy) but it predates plans 033-038: it does not mention the unit-built machines, gearbox stages, drag
  corrections or the drawn layout, which are all defaults now. The current state is in the field comments of
  `HaloAssumptions` and in `docs/AIRCRAFT_VERSIONS.md`.
  - L7-9 is the requirement set (decision 013, revised by 017 and 026): 900 kg payload, 445 nm, 210 kt at
    10,000 ft, 13,000 ft ceiling, OGE hover at 4,000 ft, engine-out hover on one turbogenerator plus battery, stall
    at most 120 kt.
  - L13-14: the AeroSandbox turboshaft mass-power regression enters as an **explicit Opti equality** (L1303), not as
    a function evaluated inside a loop.
- **L35-37** standard library: `time` (multistart wall times), `dataclasses` (`replace` is used everywhere to make
  variants of frozen dataclasses), `Any` (fields that may hold an Opti variable or a number).
- **L39-42** AeroSandbox: `asb` (Opti, Atmosphere, Airfoil), its numpy shim (works on floats and CasADi), units, and
  the turboshaft regressions `power_turboshaft(mass)` and `thermal_efficiency_turboshaft(mass)`.
- **L44-83** the package, one import per model: aerodynamics (four models plus trim), stability, margins, cost,
  mission, flight point, powertrain components and topologies, requirements, vehicle items, thermal helpers, AFDD
  weights. **L86 imports `Xv15MassFactors` and never uses it** (see Findings).
- **L84-86** three example modules are imported as libraries: an example is allowed to import another example, the
  package never imports an example.
- **L88 `fuel_factor = 1.1`**: fuel loaded = 1.1 x fuel burnt (a 10 % fuel reserve, L1302). The 20 min loiter is a
  separate reserve flown inside the mission.
- **L89-91 SOC levels.** `soc_take_off = 0.95` (start of the mission and the pack's `max_soc`), `soc_minimum = 0.30`
  (the mission floor, where the engine-out and failure hovers start), `soc_emergency_floor = 0.10` (the pack's
  `min_soc` and the floor at the end of the 60 s contingencies). The mission therefore uses the window
  0.95 → 0.30 and the contingency the window 0.30 → 0.10 (decision 013 table, "Engine-out hover").
- **L92 `scale_objective_cost_usd = 100`**: the cost objective is divided by 100 so it is of order 10. The comment
  records the empirical reason: IPOPT converged from the mass optimum with 100 and not with 1,000.

In [3]:
print("fuel_factor", fuel_factor, "| SOC take-off / floor / emergency:", soc_take_off, soc_minimum, soc_emergency_floor)
print(f"mission window {soc_take_off - soc_minimum:.2f}, contingency window {soc_minimum - soc_emergency_floor:.2f}")
print(f"fuel loaded {ref.mass_fuel_kg:.2f} kg = {ref.mass_fuel_kg / ref.mass_fuel_burnt_kg:.6f} x burnt {ref.mass_fuel_burnt_kg:.2f} kg")
check("fuel loaded = fuel_factor x fuel burnt (L1302)", abs(ref.mass_fuel_kg - fuel_factor * ref.mass_fuel_burnt_kg) < 1e-4)
battery = build_halo_battery(d, a)
check("the pack window is the emergency floor to the take-off SOC (L476)",
      battery.min_soc == soc_emergency_floor and battery.max_soc == soc_take_off)
import inspect
check("Xv15MassFactors is imported but never used in halo_sizing.py",
      inspect.getsource(halo).count("Xv15MassFactors") == 1)

fuel_factor 1.1 | SOC take-off / floor / emergency: 0.95 0.3 0.1
mission window 0.65, contingency window 0.20
fuel loaded 936.44 kg = 1.100000 x burnt 851.31 kg
PASS  fuel loaded = fuel_factor x fuel burnt (L1302)
PASS  the pack window is the emergency floor to the take-off SOC (L476)
PASS  Xv15MassFactors is imported but never used in halo_sizing.py


---
## 2. Equipment items

In [4]:
show_source("examples/halo_sizing.py", 95, 130)

# examples/halo_sizing.py  lines 95-130
  95  @dataclass(frozen=True)
  96  class EquipmentItem:
  97      """One line of the fixed-equipment build-up; negative masses are removals."""
  98      label: str
  99      mass_kg: float
 100      source: str
 101  
 102  
 103  # XV-15 fixed-equipment groups (TM X-62407 sec. 3.1.2), the starting point for the Halo equipment.
 104  xv15_equipment_items = (
 105      EquipmentItem("electrical", 396 * u.lbm, "TM X-62407 sec. 3.1.2"),
 106      EquipmentItem("instrumentation", 91 * u.lbm, "TM X-62407 sec. 3.1.2"),
 107      EquipmentItem("heating and air conditioning", 100 * u.lbm, "TM X-62407 sec. 3.1.2"),
 108      EquipmentItem("miscellaneous (furnishings, equipment)", 436 * u.lbm, "TM X-62407 sec. 3.1.2"),
 109  )
 110  # Uncrewed changes, itemised (plan 024). Until Tier 20 the same 587 lb was carried as one number, with the
 111  # heating group relabelled as autonomy; the total is unchanged.
 112  uncrewed_equipment_adjustments = (
 113   

- **L95-100 `EquipmentItem`**: a label, a mass in kg (negative for a removal) and a source string. Frozen, so a list
  of items is a constant.
- **L104-109** the four XV-15 fixed-equipment groups from NASA TM X-62407 sec. 3.1.2, entered in lb and converted
  with `u.lbm`: electrical 396, instrumentation 91, heating and air conditioning 100, miscellaneous 436 lb.
- **L112-123** the uncrewed changes (plan 024, decision 024): remove the crew ECS (-100 lb), the ejection seats
  (-230 lb, from Harris NASA/SP-2015-215959), the rest of the furnishings (-(436-230) = -206 lb), add 100 lb of
  autonomy and mission avionics (an assumed allocation). L110-111: until Tier 20 the same total was one number.
  L121-122 say what is *not* removed: cockpit controls and crew-station structure stay inside the flight-control and
  fuselage calibration factors (no public breakdown).
- **L124** the Halo list is the concatenation; **L127-128** the total is a plain sum. It is evaluated once, as the
  default of `HaloAssumptions.mass_equipment_kg` (L220).

In [5]:
for item in halo_equipment_items:
    print(f"{item.mass_kg / u.lbm:+7.0f} lb  {item.label}")
xv15_lb = mass_equipment_from_items_kg(xv15_equipment_items) / u.lbm
total_lb = mass_equipment_from_items_kg(halo_equipment_items) / u.lbm
print(f"XV-15 groups {xv15_lb:.0f} lb, Halo {total_lb:.0f} lb = {total_lb * u.lbm:.2f} kg")
check("XV-15 fixed equipment groups total 1,023 lb", abs(xv15_lb - 1023) < 1e-9)
check("Halo fixed equipment = 587 lb, the HaloAssumptions default (L220)",
      abs(total_lb - 587) < 1e-9 and HaloAssumptions().mass_equipment_kg == mass_equipment_from_items_kg(halo_equipment_items))

   +396 lb  electrical
    +91 lb  instrumentation
   +100 lb  heating and air conditioning
   +436 lb  miscellaneous (furnishings, equipment)
   -100 lb  remove crew environmental control (heating, ventilation, air conditioning)
   -230 lb  remove ejection seats
   -206 lb  remove other furnishings and miscellaneous crew equipment
   +100 lb  add autonomy and mission avionics (flight computers, sensors, datalinks)
XV-15 groups 1023 lb, Halo 587 lb = 266.26 kg
PASS  XV-15 fixed equipment groups total 1,023 lb
PASS  Halo fixed equipment = 587 lb, the HaloAssumptions default (L220)


---
## 3. `HaloRequirements`

In [6]:
show_source("examples/halo_sizing.py", 131, 171)

# examples/halo_sizing.py  lines 131-171
 131  @dataclass(frozen=True)
 132  class HaloRequirements:
 133      # 900 kg (plan 026): with the AFDD tiltrotor wing the equivalent-circuit pack carries up to 959 kg at 210 kt.
 134      # 780 kg with the Raymer wing (plan 022, requirements_plan022); 900 kg before that (requirements_tier16).
 135      mass_payload_kg: float = 900.0
 136      range_m: float = 445 * 1852.0
 137      altitude_cruise_m: float = 10000 * u.foot
 138      # 210 kt (plan 017): the most the fixed 2 x 1,120 hp turboshafts sustain with 900 kg over 445 nm (max
 139      # payload 935 kg); 250 kt is infeasible at any size with these engines. Pending the author's choice.
 140      velocity_max_m_s: float = 210 * u.knot
 141      altitude_hover_m: float = 4000 * u.foot
 142      thrust_to_weight_hover: float = 1.05
 143      altitude_ceiling_m: float = 13000 * u.foot
 144      climb_rate_ceiling_m_s: float = 0.5
 145      climb_rate_m_s: float = 7.5
 146      velocity_climb

Every field is a number in SI; the comments carry the decision history.

| Field | Default | Meaning and decision |
|---|---|---|
| `mass_payload_kg` L135 | 900 kg | 900 kg from decision 013; 780 kg in plan 022 (the ECM pack did not close at 900 with the Raymer wing); back to 900 kg with the AFDD tiltrotor wing in plan 026 |
| `range_m` L136 | 445 nm | the XV-15's range (decision 013); ground distance of the whole mission |
| `altitude_cruise_m` L137 | 10,000 ft | cruise, loiter and the max-speed point altitude |
| `velocity_max_m_s` L140 | 210 kt | decision 017: the most two fixed 1,120 hp turboshafts sustain with 900 kg over 445 nm (max payload 935 kg); 250 kt infeasible at any size. It is also the upper bound of the cruise and loiter speed variables (L1179, L1181) |
| `altitude_hover_m`, `thrust_to_weight_hover` L141-142 | 4,000 ft, 1.05 | the OGE hover requirement point |
| `altitude_ceiling_m`, `climb_rate_ceiling_m_s` L143-144 | 13,000 ft, 0.5 m/s | the ceiling point (residual climb at 80 m/s) |
| `climb_rate_m_s`, `velocity_climb_m_s` L145-146 | 7.5 m/s, 80 m/s | the climb requirement point at 1,000 m (L164-165) |
| `velocity_stall_max_m_s`, `cl_max` L147-148 | 120 kt, 1.5 | stall at or below 120 kt at MTOM: a lift margin at L1266/L1274; `cl_max` also goes to the aerodynamics (L610, L615) |
| `duration_loiter_s` L149 | 1,200 s | 20 min reserve loiter segment |
| `duration_hover_s` L150 | 60 s | take-off and landing hovers, and the duration of the hover requirement point with the thermal model |
| `duration_engine_out_hover_s` L151 | 60 s | the engine-out hover and every failure hover |
| `hover_hot_day` ... L154-158 | on, 4,000 ft, 95 °F, 1.05, 60 s | Tier 16 (decision 020): a hot/high OGE hover at the destination, at the end-of-mission mass and SOC |

**L160-169 `requirement_set()`** turns the fields into the package's `RequirementSet` (C12): four capability points
(hover, climb, max speed, ceiling) whose `flight_conditions()` become flight points at MTOM (L1204-1207). The hover
requirement takes the `HoverRequirement` default SOC (0.8, `FlightCondition.soc`), which is not exposed here
(C12 Findings).

In [7]:
for c in req.requirement_set().flight_conditions():
    print(f"{c.label:<10} mode {c.mode:<8} h {c.altitude_m:7.1f} m  V {c.velocity_m_s:6.2f} m/s  T/W {c.thrust_to_weight}  "
          f"climb {c.climb_rate_m_s} m/s  soc {c.soc}")
from examples.xv15_hot_day import temperature_offset_K
offset_K = temperature_offset_K(req.altitude_hover_hot_m, req.temperature_hover_hot_K)
print(f"hot day: 95 F = {req.temperature_hover_hot_K:.2f} K, ISA offset at 4,000 ft {offset_K:+.2f} K")
check("range 445 nm and max speed 210 kt in SI", abs(req.range_m - 824140.0) < 1e-6 and abs(req.velocity_max_m_s - 108.0333) < 1e-3)
check("hot day 4k/95 is ISA + 27.7 K", abs(offset_K - 27.66) < 0.01)

hover      mode hover    h  1219.2 m  V   0.00 m/s  T/W 1.05  climb 0.0 m/s  soc 0.8
climb      mode airplane h  1000.0 m  V  80.00 m/s  T/W 1.0  climb 7.5 m/s  soc 0.8
max_speed  mode airplane h  3048.0 m  V 108.03 m/s  T/W 1.0  climb 0.0 m/s  soc 0.8
ceiling    mode airplane h  3962.4 m  V  80.00 m/s  T/W 1.0  climb 0.5 m/s  soc 0.8
hot day: 95 F = 308.15 K, ISA offset at 4,000 ft +27.66 K
PASS  range 445 nm and max speed 210 kt in SI
PASS  hot day 4k/95 is ISA + 27.7 K


---
## 4. `HaloAssumptions`, field by field

143 fields. They are grouped below in the order of the file, which is roughly the order of the tiers. Each switch
keeps the simpler model available (the project rule "the simplest model is always kept"), and every switch whose
default changed names the plan that changed it. The named earlier baselines (section 13) are built by flipping these
switches back.

In [8]:
show_source("examples/halo_sizing.py", 172, 229)

# examples/halo_sizing.py  lines 172-229
 172  @dataclass(frozen=True)
 173  class HaloAssumptions:
 174      count_rotors: int = 2
 175      count_turbogenerators: int = 2
 176      count_blades: int = 3
 177      solidity: float = 0.089                       # XV-15
 178      speed_tip_m_s: float = 740 * u.foot           # XV-15 hover tip speed, also the bound
 179      frequency_coning_per_rev: float = 1.55        # XV-15 reading; the rotor factor calibrates it
 180      figure_of_merit: float = 0.67                 # XV-15 calibrated (Tier 10b); actuator-disk rotor only
 181      coefficient_airplane: float = 0.87            # assumed; actuator-disk rotor only
 182      # Tier 12: momentum + profile rotor calibrated on JVX (rotor speed, solidity and tip speed matter); False
 183      # restores the Tier 10c actuator disk with the two constant coefficients above.
 184      rotor_speed_physics: bool = True
 185      # Plan 017 (decided 2026-10-03): off-the-shelf turboshafts, power fi

**Configuration and rotor (Tiers 10-12).**
- **L174-176** two rotors, two turbogenerators, three blades. Counts are Python ints: structure, not variables.
- **L177-181** XV-15 rotor values. `solidity` and `speed_tip_m_s` are used only when `rotor_speed_physics=False`;
  otherwise they are design variables (L1144-1152) and `speed_tip_m_s` is only a fallback. `frequency_coning_per_rev`
  feeds the AFDD rotor-group weight (L700-701). `figure_of_merit` 0.67 and `coefficient_airplane` 0.87 are the two
  constant coefficients of the actuator-disk rotor (Tier 10b calibration).
- **L184 `rotor_speed_physics=True`** (Tier 12, decision 016): the momentum + profile rotor calibrated on JVX. False
  restores the Tier 10c actuator disk (used by `assumptions_tier11a`).

**Engines and energy (Tiers 12b-13).**
- **L187 `power_rated_turboshaft_fixed_W = 1120 hp`** (decision 017): off-the-shelf engines from the user-supplied
  GASP deck; the rating is a number, so `power_rated_turboshaft_W` in the design is not a variable (L1116). `None`
  restores the rubber engine (`assumptions_tier12`).
- **L190 `hybridization_electric_min=-1`** (decision 017): a negative battery share is allowed, so the generators can
  recharge the pack on free-split segments. 0 forbids in-flight charging.
- **L193 `soc_floor_every_segment=True`**: the 0.30 floor holds at every segment end, because the engine-out reserve
  is assessed from it (L1198).
- **L196-201 Tier 13 machines** (decision 018): `machine_mass_by_torque=True` makes motor speed, rotor gear ratio
  and generator speed design variables (L1130-1143). 15 N m/kg integrated torque density (magniX class), a 10 kW/kg
  specific-power cap, a step-up generator gearbox from the engine output shaft at 1,210 rpm (the deck's
  propeller rpm), no direct drive.
- **L202 `mach_tip_hover_max = 0.70`**: upper bound of the design tip speed at sea level (L1152). JVX was tested at
  0.68-0.73. This bound is **active** at the baseline (section 5).
- **L203** 7 % hover download (XV-15), used only by the `"simple"` aerodynamics; the buildup computes its own.
- **L204-206** fallback gear ratio and peak-efficiency speeds when the Tier 13 variables are off.

**Geometry (Tier 10a, plan 037).**
- **L207** aspect ratio 6.12 (XV-15), unless `free_aspect_ratio_wing`.
- **L210 `length_fuselage_m = 11.0`** (decision 037, "11 m, as drawn"; 42.1 ft before). **L211** 5.5 ft width.
  **L214-215** a 2.0 m deep super-ellipse with exponent 3.2 (decision 037: "archer halo is not pressurized"); `None`
  is the round XV-15 section.
- **L216 `clearance_rotor_fuselage_m = 0.3`** (XV-15 about 1 ft): the tip-to-fuselage gap in the rotor-radius margin
  (L1275), the **most expensive active margin** of the problem (section 6).
- **L217** horizontal-tail root leading edge at 9.8 m (11.4 m before plan 037). **L218** 0.8 m2 misc drag area for
  the `"simple"` aerodynamics only. **L219** 4.5 ultimate load factor (XV-15). **L220** equipment from section 2.
  **L221** 2 x 95 ft2 nacelle wetted area. **L222-223** the constant battery's resistance-energy product and
  mass smoothing (Tier 1 battery).
- **L226 `part_power_model`**: the 1,120 hp GASP deck fuel curve (decision 014); `field(default_factory=...)` because
  a dataclass default must not be a shared mutable object.
- **L229 `temperature_lapse=True`** (Tier 16, decision 020): the turboshaft lapse in density and temperature.

In [9]:
show_source("examples/halo_sizing.py", 230, 308)

# examples/halo_sizing.py  lines 230-308
 230      # ---- Tier 17 battery (plan 021) ----
 231      # "ecm": EquivalentCircuitBattery, Samsung INR21700-50G OCV and resistance shape (Paudel et al. 2025);
 232      # "constant": the Tier 1 constant-OCV Battery sized by energy and power (the reference for Tiers 10c-16).
 233      # The ECM pack is the reference from plan 022 (it cannot carry 900 kg at 210 kt on the fixed engines).
 234      battery_model: str = "ecm"
 235      # Decision 2026-10-02: resistance / factor and current rating x factor (more power-dense 50G-shaped cell).
 236      # 5: 10C continuous; below about 5 battery power sizes the pack, above about 8 the reserve energy does (plan 021).
 237      factor_power_density_battery: float = 5.0
 238      count_series_battery: int = 210               # 756 V nominal; 525-882 V window inside the 400-900 V machines
 239      factor_capacity_ageing_battery: float = 0.8   # end of life: 80 % of rated capacity (assumed)
 240      fac

**Battery (Tier 17, decisions 021 and 022).**
- **L234 `battery_model="ecm"`**: the equivalent-circuit pack (Samsung INR21700-50G shape) is the reference from plan
  022. `"constant"` is the Tier 1 constant-OCV battery; it remains a starting-point model (`constant_battery`).
- **L237 `factor_power_density_battery = 5`**: resistance divided by 5 and current rating times 5 relative to the 50G
  cell, a power-dense cell of the same shape: 10C continuous (decision 021).
- **L238 `count_series_battery = 210`**: 756 V nominal, 525-882 V window (checked below).
- **L239-242** end-of-life ageing (80 % capacity, +50 % resistance), a 25 °C managed pack, 70 % cell mass fraction.
- **L245-246** subdivision of each mission segment (`(1, 1, 4, 1, 1, 1)`: the cruise is split in four so OCV and
  resistance follow SOC) and 3 points in each 60 s contingency hover (the voltage sag near SOC 0.10 is seen).

**Electrical layer (Tier 15, decision 023), off by default.**
- **L251 `electrical_layer=False`**: True inserts inverters, active rectifiers, DC feeders and protection between
  machines, battery and bus, and sizes the machines as bare machines.
- **L255-256** bare-machine figures chosen so the integrated Tier 13 numbers are recovered (L252-254 shows the
  arithmetic). **L257-265** inverter, derating, routing and feeder assumptions (NEC-style 125 %); aluminium cable.
- **L267-270** an optional DC/DC converter regulating the bus at 800 V.

**AFDD tiltrotor wing (Tier 20, decisions 024, 026, 038).**
- **L276 `wing_weight_model="afdd_tiltrotor"`** (decision 026): the NDARC tiltrotor wing (stiffness for whirl flutter,
  jump take-off strength) with its own XV-15 factor and a design rotor speed (L1153-1158). `"raymer"` is the
  Raymer GA wing (the reference until plan 026).
- **L278-280** XV-15 symmetric wing frequencies per rev (torsion 1.09, beam 0.43, chord 0.83): the placement targets
  of the whirl-flutter margins (L1290-1293).
- **L281-282** 23 % thick NACA 2423, graphite-epoxy.
- **L286-289** (decision 038) `ratio_radius_gyration_pylon=None`: pylon pitch inertia built from the tip components
  (L795-802); a number is the XV-15 ratio 0.222.
- **L292-295** (decision 038) `ratio_depth_spar_cap=None`: spar-cap separation from the airfoil depth at the spars
  (L806-808, about 0.83 for NACA 2423); 1.0 is NDARC. 1 mm minimum torque-box gauge (0: NDARC).
- **L299 `turbogenerators_on_wing_tips=False`** (decision 037, "turbines are likely inside the fuselage"): the tips
  keep rotor, motor, rotor gearbox and cowling only.
- **L303-306** (decision 038) the turbogenerator station from the layout: half an engine length behind the rear
  spar, 0.35 m below the fuselage top. Numbers restore the plan 037 placement.
- **L307-308** jump take-off load factor 2, and the smoothing of the AFDD `max(0, .)` steps for IPOPT.

In [10]:
show_source("examples/halo_sizing.py", 309, 368)

# examples/halo_sizing.py  lines 309-368
 309      # ---- Tier 21 aerodynamics (plan 025) ----
 310      # "simple": SimpleAerodynamics with `drag_area_misc_m2` and the constant download (the reference until plan 027);
 311      # "buildup": AeroSandbox AeroBuildup plus Scholz interference, the misc. drag area below, blown wing and the
 312      # geometric hover download; "scholz": the Scholz level-0 hand check (linear lift, no blown wing).
 313      aerodynamics_model: str = "buildup"   # plan 027 (decided 2026-10-04); "simple" until then
 314      length_nacelle_m: float = 9.0 * u.foot        # assumed; with the diameter, ~95 ft2 wetted (cowling mass)
 315      diameter_nacelle_m: float = 3.3 * u.foot
 316      drag_area_misc_buildup_m2: float = 3.00 * u.foot**2  # XV-15 "fuselage fittings & fixtures" (NDARC, Johnson 2010)
 317      blown_wing: bool = True                       # "buildup": rotor slipstream increments in airplane mode
 318      # ---- Tier 22 design-space practice (

**Aerodynamics (Tier 21, decisions 025 and 027).**
- **L313 `aerodynamics_model="buildup"`** (decision 027): AeroSandbox AeroBuildup plus Scholz interference, the
  3 ft2 XV-15 "fittings and fixtures" drag area (L316), the blown wing (L317) and the geometric hover download.
  `"simple"` was the reference until plan 027; `"scholz"` is the hand check, and the `scholz_aero` starting point.
- **L314-315** nacelle 9 ft x 3.3 ft (assumed).

**Design-space practice (Tier 22, decision 029).**
- **L322-329** three freed trades, all off: cruise altitude (bounds 2,000-13,000 ft), wing aspect ratio (4-12),
  reserve SOC (0.20-0.90). Each flag turns a fixed value into an Opti variable (L1160-1174).
- **L331** the cost model (`mission/cost.py`, C12), used by `objective="cost"` and reported for every result.

**Drag corrections (plan 034, made default by decision 035).**
- **L335 `drag_corrections=True`**: excrescence factors 1.27 (buildup) / 1.17 (Scholz), calibrated so the XV-15
  components match NDARC's 6.25 ft2; the flat 2 % trim drag (L338) is used only by Scholz. **L339-340** the tail
  efficiency 0.9 and dihedral 0 for the plan 036 trim drag from the tail load.

**Fuselage mass (decision 037).** **L346 `mass_factor_fuselage=1.70`**: raw Raymer GA (unpressurized) x 1.70, anchored
to the plan 031 layout estimate of the uncrewed fuselage (530 kg = 1.73 x raw Raymer). `None` uses the XV-15 group
factor 2.07 (a crewed fuselage).

**Thermal (Tier 19, decisions 028 and 030).**
- **L353 `thermal_model=True`** (decision 030): every loss becomes a heat load; one ram-air exchanger, sized by a
  design-variable rating (L1109), adds mass, cooling drag in airplane mode and fan power in hover; motors, generators
  and the pack carry a lumped temperature through each segment so short peaks may exceed the continuous rating. The
  rotor gearbox then gets its own rating variable (L1112).
- **L354-361** exchanger at 1 kW/kg at a 40 K reference difference, 60 °C water-glycol, effectiveness 0.8, 1 kPa air
  pressure drop, fan efficiency 0.6; machines 500 J/(kg K) and 150 °C mean winding limit.
- **L364-365** the pack: 1,000 J/(kg K), 60 °C limit, its own 25 °C loop (no chiller modelled, L362-363).
- **L368** gearbox heat goes to their own oil coolers (inside the AFDD drive weights), so it is reported but not sent
  to the exchanger.

In [11]:
show_source("examples/halo_sizing.py", 369, 426)

# examples/halo_sizing.py  lines 369-426
 369      # ---- Tier 18 redundancy (plan 032) ----
 370      # True: `count_lanes_motor` lane motors per rotor (each 1/N of the torque, on a combining gearbox input),
 371      # `count_buses` cross-strapped buses joined by normally open ties (contactor + cable), `count_strings_battery`
 372      # isolated pack strings (a contactor each), and the failure hovers below as extra points of the sizing problem.
 373      # True is the default from plan 035. False: the single-lane aircraft. With True, all counts 1 and no failure cases the
 374      # problem is the same as with False.
 375      redundancy: bool = True   # plan 035 default (decided 2026-10-04); False before
 376      count_lanes_motor: int = 2
 377      count_buses: int = 2
 378      count_strings_battery: int = 2
 379      length_cable_bus_tie_m: float = 2.0              # bus to bus in the fuselage (assumed)
 380      # Failure hovers: 60 s at MTOM from the SOC floor down to the eme

**Redundancy (Tier 18, decision 032, made default by decision 035).**
- **L375 `redundancy=True`**: 2 lane motors per rotor on a combining gearbox input (L376), 2 cross-strapped buses
  joined by normally open ties (L377, L379), 2 isolated battery strings (L378).
- **L382-386** the failure hovers: lane out, bus out, string out on; the two double failures off. They are extra
  points of the same problem, not separate analyses.
- **L373-374** the limiting case: redundancy on with all counts 1 and no failure cases is the same problem as off.

**Machines (decision 033, units from decision 035).**
- **L394 `machine_mass_model="units"`** (decision 035, author: "best in class. no rubber motors."): each machine is a
  whole number of units of a real product. `"torque_density"` (Tier 13 rubber machine) and `"database"` (a speed-
  dependent torque density fitted to `data/machines/aerospace_motors.csv`, L395-397) remain.
- **L402 `gearbox_stages=True`**: explicit stage count with per-stage mass and efficiency. **L405** the *relaxed*
  (softplus) stage count: the staircase version (`staircase=True`) solves on the fast set but not through the
  AeroBuildup and thermal starts (L403-404). **L406** the rotor gear ratio may reach 40 (slow-motor trades).
- **L413-416** motor unit: Evolito D1500 2x3, 40 kg, 1,500 N m peak, 1,400 N m x 1,800 rpm = 264 kW continuous,
  2,500 rpm max. **L418-421** generator unit: Helix SPX242, 31.2 kg, 470 N m, 315 kW, 17,000 rpm.
- **L422-424** `count_units_motor/generator=None` and `integer_units=True` trigger the relax → ceil → re-solve logic
  of `solve_halo_sizing` (L1024-1033). The baseline's final assumptions have the counts fixed (2 and 3).

In [12]:
default = HaloAssumptions()
print("fields:", len(fields(HaloAssumptions)))
switches = dict(rotor_speed_physics=True, power_rated_turboshaft_fixed_W=1120 * u.hp, hybridization_electric_min=-1.0,
                soc_floor_every_segment=True, machine_mass_by_torque=True, temperature_lapse=True, battery_model="ecm",
                electrical_layer=False, dcdc_converter=False, wing_weight_model="afdd_tiltrotor",
                turbogenerators_on_wing_tips=False, aerodynamics_model="buildup", free_altitude_cruise=False,
                free_aspect_ratio_wing=False, free_soc_reserve=False, drag_corrections=True, mass_factor_fuselage=1.70,
                thermal_model=True, redundancy=True, machine_mass_model="units", gearbox_stages=True,
                integer_units=True)
for name, value in switches.items():
    print(f"  {name:<32} {getattr(default, name)!r}")
check("every switch default is as described", all(getattr(default, k) == v for k, v in switches.items()))
diff = [f.name for f in fields(HaloAssumptions) if getattr(default, f.name) != getattr(a, f.name)]
print("baseline_assumptions() differs from the defaults only in:", diff)
check("the final (integer) assumptions differ from the defaults only in the unit counts",
      diff == ["count_units_motor", "count_units_generator"])
window = bus_voltage_window(default)
print(window)
check("210 cells in series: 756 V nominal, 525-882 V window, 900 V derated inverter limit",
      (round(window.voltage_nominal_V), window.voltage_min_V, round(window.voltage_max_V, 6), window.voltage_max_inverter_V)
      == (756, 525.0, 882.0, 900.0))
check("motor unit continuous power 1,400 N m x 1,800 rpm = 264 kW", abs(default.power_continuous_unit_motor_W - 263894.0) < 1)

fields: 143
  rotor_speed_physics              True
  power_rated_turboshaft_fixed_W   835183.6880808958
  hybridization_electric_min       -1.0
  soc_floor_every_segment          True
  machine_mass_by_torque           True
  temperature_lapse                True
  battery_model                    'ecm'
  electrical_layer                 False
  dcdc_converter                   False
  wing_weight_model                'afdd_tiltrotor'
  turbogenerators_on_wing_tips     False
  aerodynamics_model               'buildup'
  free_altitude_cruise             False
  free_aspect_ratio_wing           False
  free_soc_reserve                 False
  drag_corrections                 True
  mass_factor_fuselage             1.7
  thermal_model                    True
  redundancy                       True
  machine_mass_model               'units'
  gearbox_stages                   True
  integer_units                    True
PASS  every switch default is as described
baseline_assumptions() dif

---
## 5. `HaloDesign`

In [13]:
show_source("examples/halo_sizing.py", 427, 457)

# examples/halo_sizing.py  lines 427-457
 427  @dataclass(frozen=True)
 428  class HaloDesign:
 429      """Every sized quantity; fields may be Opti variables or numbers."""
 430      x_le_wing_m: Any
 431      area_wing_m2: Any
 432      area_horizontal_tail_m2: Any
 433      area_vertical_tail_m2: Any
 434      torque_peak_motor_Nm: Any
 435      torque_peak_generator_Nm: Any
 436      power_rated_turboshaft_W: Any
 437      mass_turboshaft_bare_kg: Any
 438      power_max_discharge_battery_W: Any
 439      energy_capacity_battery_J: Any
 440      area_disk_m2: Any
 441      mass_fuel_kg: Any
 442      solidity: Any = None                          # None: the assumption value (fixed)
 443      speed_tip_m_s: Any = None                     # design (hover) tip speed; None: the assumption value
 444      speed_peak_motor_rad_s: Any = None            # Tier 13 variables; None: the assumption values
 445      reduction_ratio: Any = None
 446      speed_peak_generator_rad_s: Any = None
 4

- **L429** "fields may be Opti variables or numbers": the same dataclass carries the symbolic design inside the solve
  and the numeric design in a result (L1341-1342 rebuilds it from `solution.value`). `build_halo_aircraft` cannot
  tell the difference, which is how the cached baseline can be rebuilt numerically in every notebook.
- **L430-441** always present: wing position and area, tail areas, motor and generator torque at peak efficiency
  (McDonald Q-hat, C02), turboshaft rating and bare mass, battery power and energy, disk area per rotor, fuel.
- **L442-455** optional fields; `None` means "use the assumption value" (or, for the battery with `"ecm"`, "derived
  from the pack"). Each optional field belongs to one tier switch: solidity and tip speed (Tier 12), motor speed,
  gear ratio and generator speed (Tier 13), parallel strings (Tier 17), wing design rotor speed (Tier 20), the three
  freed trades (Tier 22), exchanger and gearbox ratings (Tier 19).

In [14]:
for f in fields(HaloDesign):
    value = getattr(d, f.name)
    print(f"  {f.name:<32} {value if value is None else f'{value:.6g}'}")
check("the three Tier 22 freed trades are off in the baseline (None)",
      d.altitude_cruise_m is None and d.aspect_ratio_wing is None and d.soc_reserve is None)
check("turboshaft rating is the fixed 1,120 hp (a number, not a variable)", abs(d.power_rated_turboshaft_W - 1120 * u.hp) < 1e-6)

  x_le_wing_m                      4.62351
  area_wing_m2                     23.2987
  area_horizontal_tail_m2          5.2463
  area_vertical_tail_m2            2.73196
  torque_peak_motor_Nm             1616.94
  torque_peak_generator_Nm         891.323
  power_rated_turboshaft_W         835184
  mass_turboshaft_bare_kg          184.207
  power_max_discharge_battery_W    850259
  energy_capacity_battery_J        2.51978e+08
  area_disk_m2                     73.36
  mass_fuel_kg                     936.443
  solidity                         0.06
  speed_tip_m_s                    238.206
  speed_peak_motor_rad_s           204.45
  reduction_ratio                  4.52847
  speed_peak_generator_rad_s       580.457
  count_parallel_battery           22.9527
  speed_rotor_wing_design_rad_s    56.2811
  altitude_cruise_m                None
  aspect_ratio_wing                None
  soc_reserve                      None
  power_rated_heat_exchanger_W     126904
  power_rated_gearbox_W   

---
## 6. Component builders (brief; C03, C06, C07, C08 go deep)

In [15]:
show_source("examples/halo_sizing.py", 458, 506)

# examples/halo_sizing.py  lines 458-506
 458  def battery_thermal(assumptions):
 459      """Tier 19: the pack's lumped thermal model (its own loop at the managed cell temperature)."""
 460      a = assumptions
 461      if not a.thermal_model:
 462          return {}
 463      return dict(thermal_model=LumpedThermalModel(a.specific_heat_battery_J_kg_K, a.temperature_max_battery_C,
 464                                                   a.temperature_cell_battery_C))
 465  
 466  
 467  def build_halo_battery(design, assumptions=HaloAssumptions()):
 468      a, d = assumptions, design
 469      if a.battery_model == "ecm":
 470          return EquivalentCircuitBattery(count_series=a.count_series_battery, count_parallel=d.count_parallel_battery,
 471                                          factor_power_density=a.factor_power_density_battery,
 472                                          temperature_cell_C=a.temperature_cell_battery_C,
 473                                          facto

- **L458-464 `battery_thermal`**: with the thermal model the pack gets a `LumpedThermalModel` (heat capacity, 60 °C
  limit, its 25 °C loop) as a keyword argument; without it an empty dict, so `**battery_thermal(a)` adds nothing.
- **L467-483 `build_halo_battery`**: `"ecm"` builds `EquivalentCircuitBattery` from the series count (assumption)
  and the **parallel count (design variable)**; energy and power are outputs of the pack. `"constant"` builds the
  Tier 1 battery from two design variables, energy and discharge power, with resistance = (resistance-energy
  product) / energy and a charge rating of half the discharge rating. Both use the SOC window 0.10-0.95. Covered in
  C03.
- **L486-504 `BusVoltageWindow` / `bus_voltage_window`**: the bus voltages implied by the pack (series count x cell
  nominal/min/max: 3.6/2.5/4.2 V) or, with the DC/DC converter, the regulated voltage at all three; and the derated
  inverter limit (blocking x 0.75). Used by the electrical layer, the redundancy ties and the bus-voltage options.

In [16]:
show_source("examples/halo_sizing.py", 507, 605)

# examples/halo_sizing.py  lines 507-605
 507  def power_electric_rated_W(machine):
 508      """Electrical rating of a machine's converter and feeder: shaft rating / peak machine efficiency."""
 509      return machine.power_rated_W / machine.loss_model.efficiency_peak
 510  
 511  
 512  def build_halo_electrical(motor, generator, battery, span_m, requirements=HaloRequirements(),
 513                            assumptions=HaloAssumptions()):
 514      """Tier 15 feeders: inverters at machine rating, cables and protection sized at rated power and the
 515      lowest bus voltage (battery: its discharge current rating), insulation for the highest bus voltage at the
 516      ceiling."""
 517      a, window = assumptions, bus_voltage_window(assumptions)
 518      cell = inr21700_50g_cell()
 519      pack_min_V = a.count_series_battery * cell.voltage_min_V
 520      pack_max_V = a.count_series_battery * cell.voltage_max_V
 521      if isinstance(battery, EquivalentCircuitBattery):
 522 

- **L507-509 `power_electric_rated_W`**: shaft rating / peak machine efficiency, the electrical power a converter or
  feeder must carry.
- **L512-565 `build_halo_electrical`** (Tier 15, C06/C07): inverters at the machine's electrical rating (L530-535),
  feeders as cable + protection rated at 125 % of the source's current at the *lowest* bus voltage (L537-543; the
  125 % also keeps cable and contactor limits from duplicating the source's own limit, a degenerate pair). Feeder
  lengths: 1.25 x half span to the nacelles (L545); generators in the fuselage use the 3 m battery length (L549,
  plan 037). Optional DC/DC (L554-561). Off in the baseline.
- **L568-570 `count_lanes_motor`**: lanes per rotor, 1 without redundancy. Used everywhere a "motor" means one lane.
- **L573-603 `build_halo_redundancy`** (Tier 18, C07): one contactor per battery string at 125 % of a string's share
  of the pack current (L590-592); with more than one bus, a tie contactor and a 2 m tie cable rated at 125 % of one
  bus's share of all sources (generators' electrical rating plus pack discharge) at the lowest bus voltage
  (L594-600). Returns `RedundancyLayer`, or `None` without the option (L578-579).

In [17]:
aircraft = build_halo_aircraft(d, req, a)
instances = aircraft.powertrain.topology.instances
battery, generator = instances["battery"].component, instances["generator"].component
current_max_A = battery.get_limits().max_discharge_current_A
protection_string = instances["protection_string"].component
print(f"pack: {battery.count_parallel:.3f} parallel strings, {battery.energy_capacity_J / 3.6e6:.2f} kWh, "
      f"{battery.power_max_discharge_W / 1e3:.1f} kW, max current {current_max_A:.1f} A")
print(f"string contactor rating {protection_string.max_current_A:.1f} A = 1.25 x {current_max_A:.1f} A / 2")
check("ECM pack energy equals the reported battery energy", abs(battery.energy_capacity_J / 3.6e6 - ref.energy_battery_kWh) < 1e-6)
check("string contactor at 125 % of half the pack current (L591-592)",
      abs(protection_string.max_current_A - 1.25 * current_max_A / 2) < 1e-9)
tie = instances["protection_bus_tie"].component
tie_A = 1.25 * (2 * power_electric_rated_W(generator) + battery.power_max_discharge_W) / (2 * 525.0)
check("bus-tie rating: 125 % of one bus's share of the sources at 525 V (L595-596)", abs(tie.max_current_A - tie_A) < 1e-9)
print("electrical layer in the baseline topology:", any(n.startswith("inverter") for n in instances))

pack: 22.953 parallel strings, 69.99 kWh, 850.3 kW, max current 1124.7 A
string contactor rating 702.9 A = 1.25 x 1124.7 A / 2
PASS  ECM pack energy equals the reported battery energy
PASS  string contactor at 125 % of half the pack current (L591-592)
PASS  bus-tie rating: 125 % of one bus's share of the sources at 525 V (L595-596)
electrical layer in the baseline topology: False


In [18]:
show_source("examples/halo_sizing.py", 606, 634)

# examples/halo_sizing.py  lines 606-634
 606  def build_halo_aerodynamics(requirements=HaloRequirements(), assumptions=HaloAssumptions()):
 607      a, r = assumptions, requirements
 608      if a.aerodynamics_model == "simple":
 609          return SimpleAerodynamics(drag_area_misc_m2=a.drag_area_misc_m2,
 610                                    download_fraction_hover=a.download_fraction_hover, cl_max=r.cl_max)
 611      if a.aerodynamics_model == "buildup":
 612          # Plan 036: trim drag from the tail load about the CG (Scholz ch. 11) replaces the flat fraction.
 613          corrections = (dict(factor_excrescence=a.factor_excrescence_buildup,
 614                              trim=TailTrim(a.efficiency_tail, a.angle_dihedral_tail_deg)) if a.drag_corrections else {})
 615          return BuildupAerodynamics(cl_max=r.cl_max, drag_area_misc_m2=a.drag_area_misc_buildup_m2,
 616                                     blown_wing=BlownWing() if a.blown_wing else None, **corrections)
 61

- **L606-621 `build_halo_aerodynamics`**: the three models (C08). With `"buildup"` and drag corrections the trim drag
  comes from the tail load about the CG (`TailTrim`, plan 036) and the excrescence factor 1.27 multiplies component
  drag; the blown wing is on. Unknown names raise (L621): no silent fallback.
- **L624-627 `ratio_reference_gear_stages`**: the XV-15 engine-to-rotor ratio (20,000 / 565 rpm = 35.4), where the
  AFDD transmission factor was calibrated.
- **L630-632 `stage_mass_ratio`**: the gearbox mass relative to the AFDD drive at the calibration ratio, so the
  stage model multiplies an already calibrated weight (decision 033).

In [19]:
aerodynamics = build_halo_aerodynamics(req, a)
print(type(aerodynamics).__name__, "| excrescence", aerodynamics.factor_excrescence, "| trim", type(aerodynamics.trim).__name__)
print(f"reference ratio {ratio_reference_gear_stages():.3f}; rotor ratio {d.reduction_ratio:.3f}: mass x "
      f"{stage_mass_ratio(a.gear_stage_model, d.reduction_ratio):.4f}, efficiency {a.gear_stage_model.efficiency(d.reduction_ratio):.4f}")
check("XV-15 calibration ratio 20,000 / 565 rpm", abs(ratio_reference_gear_stages() - 20000 / 565) < 1e-9)
check("stage mass ratio is 1 at the calibration ratio", abs(stage_mass_ratio(a.gear_stage_model, ratio_reference_gear_stages()) - 1) < 1e-12)

BuildupAerodynamics | excrescence 1.27 | trim TailTrim
reference ratio 35.398; rotor ratio 4.528: mass x 0.7355, efficiency 0.9800
PASS  XV-15 calibration ratio 20,000 / 565 rpm
PASS  stage mass ratio is 1 at the calibration ratio


---
## 7. `build_halo_aircraft` (brief; C09 covers it line by line)

In [20]:
show_source("examples/halo_sizing.py", 635, 765)

# examples/halo_sizing.py  lines 635-765
 635  def build_halo_aircraft(design, requirements=HaloRequirements(), assumptions=HaloAssumptions(), factors=None,
 636                          lapse_exponent=None):
 637      a, d = assumptions, design
 638      factors = factors if factors is not None else calibration_factors()
 639      lapse_exponent = lapse_exponent if lapse_exponent is not None else fit_lapse_exponent()
 640      ratios = dict(torque_ratio=2.5, power_ratio=1.25, speed_ratio=2.5)
 641      by_torque = a.machine_mass_by_torque
 642      if a.machine_mass_model not in ("torque_density", "database", "units"):
 643          raise ValueError(f"Unknown machine mass model '{a.machine_mass_model}'.")
 644      is_database = a.machine_mass_model == "database"
 645      if is_database and not (by_torque or a.electrical_layer):
 646          raise ValueError("The database machine mass model needs machine_mass_by_torque (speed design variables).")
 647      if a.electrical_layer:
 64

What matters for the sizing problem:

- **L638-639** calibration factors and the lapse exponent are computed once per call (C16); the solver passes them
  in (L1188) so they are not recomputed.
- **L640-677** machines. `rubber_machine` (C02) builds McDonald machines from the design's peak-efficiency speed and
  torque with fixed ratios (torque x 2.5, power x 1.25, speed x 2.5). The mass model is chosen by
  `machine_mass_model` and the electrical layer.
- **L678-689** with `"units"`, `unit_machine` (L987) replaces the limits and mass by those of n whole units; the
  McDonald point (the design variables) then only places the efficiency map (L679-682).
- **L692-694** the rotor gearbox rating: its own design variable with the thermal model, otherwise the motor rating
  times the lanes.
- **L695-701** rotor radius from the disk area, chord from solidity, design rotor speed from tip speed / radius,
  AFDD82 rotor-group mass x the XV-15 rotor factor.
- **L702-742** rotor gearbox (AFDD00 or AFDD83 x transmission factor, x the stage mass ratio and stage efficiency)
  and the generator step-up gearbox (L726-742).
- **L743-758** rotor (momentum + profile or actuator disk), battery, turboshaft (bare mass x powerplant factor,
  efficiency from the AeroSandbox regression, the deck part-power curve, the XV-15 temperature lapse).
- **L759-764** electrical and redundancy layers, then `build_series_hybrid` assembles the topology (C07).

In [21]:
show_source("examples/halo_sizing.py", 766, 900)

# examples/halo_sizing.py  lines 766-900
 766      nacelles = Nacelles(mass_engines_kg=a.count_turbogenerators * d.mass_turboshaft_bare_kg,
 767                          count_engines=a.count_turbogenerators, area_wetted_m2=a.area_wetted_nacelles_m2,
 768                          x_m=0.0, z_m=1.2, mass_factor=factors.powerplant)
 769      if a.wing_weight_model == "raymer":
 770          wing_mass = dict(mass_factor=factors.wing)
 771      elif a.wing_weight_model == "afdd_tiltrotor":
 772          # Mass on one wing tip: rotor, motor and rotor gearbox, plus the turbogenerator and its nacelle section.
 773          mass_motors_tip_kg = motor.get_mass() if lanes == 1 else lanes * motor.get_mass()    # Tier 18 lanes
 774          mass_tip_kg = mass_rotors_kg / a.count_rotors + mass_motors_tip_kg + gearbox.get_mass()
 775          if electrical is not None:
 776              # Tier 15: inverters in the nacelle (one per lane with Tier 18 lanes).
 777              mass_tip_kg = mass_tip_kg 

- **L766-768 nacelles**: the AFDD82 engine section (support, air induction, cowling) from the engine count and bare
  mass.
- **L769-820 wing mass**: Raymer x factor, or the AFDD tiltrotor wing built for the tip mass (L773-788), the pylon
  radius of gyration from the tip components (L789-802, decision 038), the spar-cap depth from the airfoil (L803-808)
  and the **wing design rotor speed** (a design variable, L813-814).
- **L821-834** wing, rotor station at the wing quarter chord (L824: the hover pitch-trim constraint L1301 puts the
  CG there), mast height, turbogenerator station from the layout (L829-834).
- **L835-866** the installed powertrain items with their stations (CG). **L867-876** the ram-air exchanger.
- **L877-898** the `Aircraft`: tails (fixed aspect ratios, XV-15 factors), fuselage (factor 1.70), gear, systems,
  powertrain, payload and fuel at the rotor station, nacelles moved to the turbogenerator station (L895-896),
  equipment at 3.5 m.

In [22]:
installed = dict((item.instance_name, float(item.mass_properties.mass))
                 for item in aircraft.powertrain.get_instance_mass_properties())
worst = max(abs(installed[name] - mass_kg) for name, mass_kg in ref.powertrain_masses_kg)
print({name: round(mass_kg, 2) for name, mass_kg in installed.items()})
check("rebuilt numeric aircraft reproduces every installed powertrain mass of the result (< 1e-6 kg)", worst < 1e-6)
station = {loc.instance_name: loc.x_m for loc in aircraft.powertrain.locations}["propulsor"]
quarter_chord = d.x_le_wing_m + 0.25 * aircraft.wing.chord_root_m()
print(f"rotor station {station:.4f} m = wing quarter chord {quarter_chord:.4f} m")
check("rotor station = wing quarter chord (L824, L840)", abs(station - quarter_chord) < 1e-12)
check("rebuilt rotor radius = result's radius", abs(instances["propulsor"].component.radius_m() - ref.radius_rotor_m) < 1e-9)

{'turboshaft': 421.72, 'generator': 281.7, 'battery': 475.12, 'motor': 425.56, 'gearbox': 292.42, 'propulsor': 674.88, 'generator_gearbox': 144.17, 'protection_string': 4.46, 'protection_bus_tie': 9.13, 'cable_bus_tie': 14.89, 'heat_exchanger': 126.9}
PASS  rebuilt numeric aircraft reproduces every installed powertrain mass of the result (< 1e-6 kg)
rotor station 5.1113 m = wing quarter chord 5.1113 m
PASS  rotor station = wing quarter chord (L824, L840)
PASS  rebuilt rotor radius = result's radius


---
## 8. `halo_mission`, `HaloSizingResult`, small helpers

In [23]:
show_source("examples/halo_sizing.py", 901, 915)

# examples/halo_sizing.py  lines 901-915
 901  def halo_mission(requirements=HaloRequirements(), velocity_cruise_m_s=None, velocity_loiter_m_s=None):
 902      r = requirements
 903      climb = ClimbSegment(0.0, r.altitude_cruise_m, 6.0, 80.0, None, "climb")
 904      descent = DescentSegment(r.altitude_cruise_m, 0.0, 5.0, 80.0, None, "descent")
 905      distance_cruise_m = r.range_m - ground_distance_m(climb) - ground_distance_m(descent)
 906      return Mission((
 907          HoverSegment(r.duration_hover_s, 0.0, None, "take-off hover"),
 908          climb,
 909          CruiseSegment(distance_cruise_m, r.altitude_cruise_m, velocity_cruise_m_s, None, "cruise"),
 910          LoiterSegment(r.duration_loiter_s, r.altitude_cruise_m, velocity_loiter_m_s, None, "reserve loiter"),
 911          descent,
 912          HoverSegment(r.duration_hover_s, 0.0, None, "landing hover"),
 913      ))
 914  
 915  


- **L903-904** climb at 6 m/s and 80 m/s to the cruise altitude; descent at 5 m/s and 80 m/s. These are not the
  requirement climb (7.5 m/s at 1,000 m): the mission climb is a flown segment, the requirement is a capability
  point.
- **L905** the cruise distance is what is left of the 445 nm after the climb and descent ground distances, so the
  mission covers exactly the range. If the altitude is a variable (freed trade) this is symbolic.
- **L906-913** six segments: take-off hover, climb, cruise (speed `velocity_cruise_m_s`), 20 min loiter
  (`velocity_loiter_m_s`), descent, landing hover. The two speeds are passed in; the solver passes Opti variables
  (L1177-1181). The defaults `None` are not usable in a solve (C12 Findings).

In [24]:
from aircraft_closure.mission.segments import ground_distance_m
mission = halo_mission(req, velocity_cruise_m_s=ref.velocity_cruise_m_s, velocity_loiter_m_s=ref.velocity_loiter_m_s)
climb, descent = mission.segments[1], mission.segments[4]
print([s.label for s in mission.segments])
print(f"climb {ground_distance_m(climb) / 1852:.2f} nm + cruise {mission.segments[2].distance_m / 1852:.2f} nm + descent "
      f"{ground_distance_m(descent) / 1852:.2f} nm")
check("climb + cruise + descent ground distance = 445 nm",
      abs(ground_distance_m(climb) + mission.segments[2].distance_m + ground_distance_m(descent) - req.range_m) < 1e-6)

['take-off hover', 'climb', 'cruise', 'reserve loiter', 'descent', 'landing hover']
climb 21.88 nm + cruise 396.84 nm + descent 26.28 nm
PASS  climb + cruise + descent ground distance = 445 nm


In [25]:
show_source("examples/halo_sizing.py", 916, 1005)

# examples/halo_sizing.py  lines 916-1005
 916  @dataclass(frozen=True)
 917  class HaloSizingResult:
 918      mass_takeoff_kg: float
 919      mass_payload_kg: float
 920      mass_empty_kg: float
 921      mass_fuel_kg: float
 922      mass_fuel_burnt_kg: float
 923      design: HaloDesign
 924      energy_battery_kWh: float
 925      radius_rotor_m: float
 926      span_m: float
 927      disk_loading_kg_m2: float
 928      wing_loading_kg_m2: float
 929      power_rated_motor_W: float
 930      power_rated_generator_W: float
 931      velocity_cruise_m_s: float
 932      velocity_loiter_m_s: float
 933      lift_to_drag_cruise: float
 934      cl_cruise: float
 935      static_margin: float
 936      cn_beta_per_rad: float
 937      soc_end: float
 938      soc_after_engine_out: float
 939      closure_residual_kg: float
 940      turboshaft_mass_power_residual: float
 941      thermal_efficiency_turboshaft: float
 942      segments: tuple
 943      binding: tuple
 944      min_ma

- **L916-972 `HaloSizingResult`**: every number a notebook or test reads, all floats or tuples of floats and dicts:
  no CasADi object survives a solve, so the result pickles (the cache) and can seed another solve (`initial`).
  L918-946 are the original fields; the defaulted ones were added by later tiers, which keeps older constructors
  valid.
- **L961-962 `machine_units`** says "(motor, generator) unit counts, relaxed (what the units would need) and fixed".
  In a result from the integer solve, the first entry is **the fixed count**, not the relaxed one: L1002 evaluates
  `mass_model.count_units`, which `unit_machine` set to the fixed integer (L990-993). The relaxed counts of the first
  solve are not kept in the final result (see Findings).
- **L975-984 `count_parallel_guess`**: the initial parallel-string count. A start that already has an ECM pack keeps
  its count. A constant-battery start gets 30, except with the electrical layer, where it is energy-matched (energy /
  energy of one string), so packs of other series counts start at the same energy.
- **L987-993 `unit_machine`**: `replace` the rubber machine's max torque, rated power, max speed and mass model with
  those of n units; n is `count_units` or the relaxed count (a smooth max of torque and power need, C02).
- **L996-1003 `machine_units_record`**: `{machine: (count in this solve, fixed count or None)}`.

In [26]:
print("result fields:", len(fields(HaloSizingResult)))
print("count_parallel_guess(baseline design) =", count_parallel_guess(d, a))
print("count_parallel_guess(constant-battery design, no layer) =",
      count_parallel_guess(replace(d, count_parallel_battery=None), a))
print("baseline machine_units:", ref.machine_units)
check("in the final result machine_units holds the fixed counts twice (L1002 reads the fixed mass-model count)",
      all(n == fixed for n, fixed in ref.machine_units.values()))
relaxed = build_halo_aircraft(d, req, replace(a, count_units_motor=None, count_units_generator=None)).powertrain.topology.instances
print({name: round(float(relaxed[name].component.mass_model.count_units), 3) for name in ("motor", "generator")},
      "<- relaxed counts of the rubber ratings at the final design (meaningless there: Q-hat only places the map)")

result fields: 42
count_parallel_guess(baseline design) = 22.952681656792883
count_parallel_guess(constant-battery design, no layer) = 30.0
baseline machine_units: {'motor': (2.0, 2), 'generator': (3.0, 3)}
PASS  in the final result machine_units holds the fixed counts twice (L1002 reads the fixed mass-model count)


{'motor': 2.695, 'generator': 4.741} <- relaxed counts of the rubber ratings at the final design (meaningless there: Q-hat only places the map)


---
## 9. `solve_halo_sizing`: the dispatcher

In [27]:
show_source("examples/halo_sizing.py", 1006, 1074)

# examples/halo_sizing.py  lines 1006-1074
1006  def solve_halo_sizing(requirements=HaloRequirements(), assumptions=HaloAssumptions(), factors=None, verbose=False,
1007                        max_iter=3000, initial=None, objective="mass_takeoff", staged_start=True, stage_fallback=True):
1008      """Size the Halo: one coupled AeroSandbox solve from `initial`, or the Tier 22 starting-point strategy.
1009  
1010      `initial`: an earlier `HaloSizingResult` used as the initial guess (sensitivity studies); the problem is then
1011      solved once from it. `staged_start` (Tier 18): False drops the "strings_off" start for redundant aircraft.
1012  
1013      objective "mass_takeoff" minimizes take-off mass at the required payload; "payload" makes payload a
1014      variable and maximizes it (the aircraft is sized around fixed engines, plan 017); "cost" minimizes the
1015      cost per mission of `HaloAssumptions.cost_model` at the required payload (Tier 22).
1016  
1017      With no `init

The function decides *which* solves to run; it never writes a constraint. Branches in order:

1. **Integer units, L1024-1033** (decision 035). With unit-built machines, `integer_units` and a count left `None`:
   - L1028-1029 recursive call with `integer_units=False`: the whole strategy (multistart included) solves the
     relaxed problem, where n is a continuous smooth max;
   - L1030 round each relaxed count **up**, after subtracting 1e-3: a relaxed 2.0005 (IPOPT noise on an integer)
     stays 2, a relaxed 2.002 becomes 3;
   - L1031-1032 a count given by the caller wins (`a.count_units_motor or counts[...]`);
   - L1033 **one** re-solve with the counts fixed, warm-started from the relaxed result. Two explicit solves: no
     branch and bound, no loop. Rounding up keeps the integer design feasible for torque and power (more capacity),
     and the re-solve re-optimizes everything else around the heavier machines.
2. **L1034-1035** with redundancy, `failure_hover_cases` is called only to raise early on an impossible case.
3. **Staircase fallback, L1036-1056** (decision 033), only for `staircase=True` stage models (not the default): two
   complete candidates, the strategy's own and a solve warm-started from the relaxed-stage solution; the better
   objective is kept. The staircase landscape has one local optimum per stage count, so two starts are a cheap hedge.
4. **Multistart, L1057-1061**: no `initial`, so the Tier 22 strategy (section 12). `staged_start=False` removes the
   `strings_off` start.
5. **Strings-off fallback, L1062-1071** (decision 032): with a caller's start and battery strings, try the target
   problem from it (L1066); if IPOPT fails, solve the same aircraft without strings and string-out cases from that
   start (L1069-1071) and use that solution as the start of the target (L1072).
6. **L1072** otherwise one solve from `initial`.

`RuntimeError` is the only failure signal: `asb.Opti.solve` raises it when IPOPT does not converge, and every
fallback catches exactly that.

In [28]:
counts = {n: int(np.ceil(n - 1e-3)) for n in (2.0, 2.0005, 2.002, 2.69, 4.74)}
print("relaxed -> integer:", counts)
check("ceil(n - 1e-3): IPOPT noise on an integer does not add a unit, anything above does",
      counts == {2.0: 2, 2.0005: 2, 2.002: 3, 2.69: 3, 4.74: 5})
default = HaloAssumptions()
check("default needs the integer branch (units, integer_units, counts None)",
      default.machine_mass_model == "units" and default.integer_units and default.count_units_motor is None)
check("the staircase branch is not taken by default (relaxed stages)", not default.gear_stage_model.staircase)
check("with the final counts fixed, solve_halo_sizing(initial=...) goes to the strings-off branch (L1062)",
      a.redundancy and a.count_strings_battery > 1 and a.count_units_motor is not None)

relaxed -> integer: {2.0: 2, 2.0005: 2, 2.002: 3, 2.69: 3, 4.74: 5}
PASS  ceil(n - 1e-3): IPOPT noise on an integer does not add a unit, anything above does
PASS  default needs the integer branch (units, integer_units, counts None)
PASS  the staircase branch is not taken by default (relaxed stages)
PASS  with the final counts fixed, solve_halo_sizing(initial=...) goes to the strings-off branch (L1062)


---
## 10. `solve_halo_sizing_once`: the problem

This is the function to know by heart. It builds one `asb.Opti`, declares the design variables, builds the aircraft
from them, flies the mission and every extra point on that one aircraft, writes the closure equalities and every
margin as `>= 0`, sets one objective and calls IPOPT once.

### 10.1 Set-up and design variables

In [29]:
show_source("examples/halo_sizing.py", 1075, 1129)

# examples/halo_sizing.py  lines 1075-1129
1075  def solve_halo_sizing_once(requirements=HaloRequirements(), assumptions=HaloAssumptions(), factors=None,
1076                             verbose=False, max_iter=3000, initial=None, objective="mass_takeoff"):
1077      """One coupled solve from `initial` (None: the generic guess). Raises RuntimeError if IPOPT fails."""
1078      if objective not in ("mass_takeoff", "payload", "cost"):
1079          raise ValueError(f"Unknown objective '{objective}'.")
1080      factors = factors if factors is not None else calibration_factors()
1081      lapse_exponent = fit_lapse_exponent()
1082      a, r = assumptions, requirements
1083      aerodynamics = build_halo_aerodynamics(r, a)
1084      longitudinal, directional = LongitudinalStability(), DirectionalStability()
1085      opti = asb.Opti()
1086  
1087      # ---- Design variables -------------------------------------------------------------
1088      guess = initial.design if initial is not Non

**L1078-1086 set-up.** Objective name validated; calibration factors and lapse exponent computed once; aerodynamics
and the two stability models built (they hold no state). **L1085** one `asb.Opti()`.

**Scaling convention.** `opti.variable(init_guess, scale, lower_bound, upper_bound)` creates `x = scale * x_hat` and
lets IPOPT work on `x_hat`; when `scale` is omitted AeroSandbox uses `|init_guess|`. Bounds are written as constraints
`x / scale >= lb / scale` (they are rows of `g`, not IPOPT variable bounds), which is why they have multipliers in
the x-ray below. A good scale makes `x_hat` of order one at the optimum.

**L1088-1092 the generic guess**: a plausible light tiltrotor (18 m2 wing, 40 m2 disks, 1 MW, 1,000 kg fuel). A
warm start replaces it by the previous design (`initial.design`).

**L1093-1094 MTOM**: init 6,000 kg (or the start's), scale 1,000, lower bound 1,000 kg (keeps the mass positive
when IPOPT wanders). The objective is `MTOM / 1000` (L1329), the same scale, so the objective is of order 7.

**L1095-1098** a start sized with another lane count keeps the *total* torque per rotor: per-lane torque x
(old lanes / new lanes). This makes `redundancy_off` (1 lane) a usable start for the 2-lane problem.

**L1101-1128 the design variables** (values and bound status at the solution are in the x-ray table, 10.7):

| Variable | Init (generic) | Scale | Bounds | Why |
|---|---|---|---|---|
| `x_le_wing_m` L1101 | 5.5 m | auto (init) | 3.0-8.5 m | keeps the wing on the 11 m fuselage; the CG equality L1301 sets it |
| `area_wing_m2` L1102 | 18 m2 | 10 | 5-60 m2 | stall margin and AFDD wing |
| tail areas L1103-1104 | 4.7 m2 | auto | >= 0.5 m2 | static margin and Cn_beta |
| motor, generator Q-hat L1105-1107 | 2,500 / 2,000 N m | 1,000 | >= 100 N m | McDonald peak-efficiency torque |
| `power_rated_heat_exchanger_W` L1109-1111 | 300 kW | 1e5 | >= 1 kW | thermal only: exchanger rating |
| `power_rated_gearbox_W` L1112-1115 | 700 kW (or the start's motor rating) | 1e6 | >= 10 kW | thermal only: drive rating apart from the continuous motor rating |
| `power_rated_turboshaft_W` L1116-1118 | fixed 1,120 hp | (1e6) | (>= 50 kW) | a number unless the fixed rating is `None` |
| `mass_turboshaft_bare_kg` L1119 | 230 kg | 100 | >= 10 kg | free, tied to the rating by the regression equality L1303 |
| battery power, energy L1121-1124 | 1 MW, 50 kWh | 1e6, 1e8 | >= 10 kW, >= 1 MJ | constant battery only |
| `count_parallel_battery` L1125-1126 | 30 (see L975) | 10 | >= 1 | ECM: parallel strings, continuous |
| `area_disk_m2` L1127 | 40 m2 | 10 | 5-120 m2 | per rotor; radius limited by the span margin L1275 |
| `mass_fuel_kg` L1128 | 1,000 kg | 500 | >= 0 | tied to the burn by L1302 |

`None` fields (L1121-1126) are filled after the aircraft is built (L1191-1193): with the ECM pack, energy and power are
expressions of `count_parallel_battery`.

In [30]:
show_source("examples/halo_sizing.py", 1130, 1182)

# examples/halo_sizing.py  lines 1130-1182
1130      if a.machine_mass_by_torque:
1131          # Tier 13: motor speed, rotor gear ratio and (with a step-up gearbox) generator speed are trades.
1132          design = replace(design, speed_peak_motor_rad_s=opti.variable(
1133              init_guess=guess.speed_peak_motor_rad_s if guess.speed_peak_motor_rad_s is not None
1134              else (60.0 if a.direct_drive_rotor else 400.0), scale=100.0, lower_bound=20.0, upper_bound=2000.0))
1135          if not a.direct_drive_rotor:
1136              design = replace(design, reduction_ratio=opti.variable(
1137                  init_guess=min(guess.reduction_ratio if guess.reduction_ratio is not None else 7.0,
1138                                 a.reduction_ratio_max),
1139                  lower_bound=1.5, upper_bound=a.reduction_ratio_max))
1140          if a.generator_step_up:
1141              design = replace(design, speed_peak_generator_rad_s=opti.variable(
1142                  init_

**L1130-1143 Tier 13 machine speeds** (decision 018), with `machine_mass_by_torque`:
- `speed_peak_motor_rad_s`: init 400 rad/s (60 for direct drive), scale 100, bounds 20-2,000 rad/s;
- `reduction_ratio`: init 7 (capped at `reduction_ratio_max`), auto scale, bounds 1.5-40;
- `speed_peak_generator_rad_s`: init 600 rad/s, scale 100, bounds from the engine output speed (1,210 rpm = 126.7
  rad/s: a step-up gearbox cannot step down) to 2,500 rad/s.

**L1144-1152 Tier 12 rotor** (decision 016): `solidity` 0.06-0.14 (init 0.10), `speed_tip_m_s` 150 m/s to
0.70 x the sea-level speed of sound = 238.2 m/s (init 220, scale 100). The upper bound is the hover tip-Mach limit
of L202.

**L1153-1158 Tier 20 wing design rotor speed**: init 40 rad/s, scale 10, bounds 5-200 rad/s. The AFDD wing is built
so its modes sit at the target per-rev values of *this* speed; the whirl margins (L1287-1293) then check every
airplane-mode point's actual rotor speed against the realized frequencies. A higher design speed means a stiffer,
heavier wing (torque box and stiffness caps grow, jump caps shrink).

**L1159-1176 freed trades** (decision 029): altitude (scale 1,000), aspect ratio and reserve SOC become variables
within their bounds; L1175-1176 fall back to the fixed values. They are off in the baseline.

**L1177-1181 mission speeds**: `velocity_cruise_m_s` (init 110 m/s or the start's, scale 50, 60 m/s to 210 kt) and
`velocity_loiter_m_s` (init 80, scale 50, 55 m/s to 210 kt) are variables passed into `halo_mission`. They are the
only operational variables declared here; all the others (rotor speed, angle of attack, battery current, battery
share, generator torque, thrust, cooling drag per point) are declared inside `build_flight_point` (C11).

### 10.2 The x-ray: one warm re-solve of the final problem

`capture_opti()` patches `asb.Opti.subject_to` and `solve` for the block: it records every `subject_to` call and the
duals it returned, and selected locals of the function that called `solve()` (here `all_margins`, `mass_takeoff_kg`,
`design`, `flown`, `total`, `breakdown`). The re-solve starts from the cached baseline, so it converges to the same
point; it is used only to look inside. Both solves of this notebook are cached with `cached()` (about 4 min when the
cache is cold, under a second after).

How the cell finds things:
- the margins are the `subject_to` call whose duals list has the length of `all_margins` (L1305);
- the closure equalities are the last 4-element call before it (L1299-1304);
- a design variable's bounds are the recorded `<=` constraints that involve only that variable's symbol and a
  constant, and the first lower and first upper among them are the ones declared by `opti.variable` (later
  single-symbol constraints come from the flight points).

In [31]:
def compute_xray():
    ref, a = baseline(), baseline_assumptions()
    with capture_opti() as x:
        r = solve_halo_sizing_once(HaloRequirements(), a, initial=ref)
    sol, calls = x.solution, x.calls
    margins = x.locals["all_margins"]
    index = max(i for i, (c, d) in enumerate(calls) if isinstance(d, list) and len(d) == len(margins))
    closure_constraints, closure_duals = next(calls[i] for i in range(index - 1, -1, -1)
                                              if isinstance(calls[i][1], list) and len(calls[i][1]) == 4)
    design, flown = x.locals["design"], x.locals["flown"]
    variables = {f.name: getattr(design, f.name) for f in fields(design)}
    variables["mass_takeoff_kg"] = x.locals["mass_takeoff_kg"]
    variables["velocity_cruise_m_s"] = flown.segments[2].segment.velocity_m_s
    variables["velocity_loiter_m_s"] = flown.segments[3].segment.velocity_m_s
    rows = []
    for name, v in variables.items():
        if not isinstance(v, cas.MX):
            rows.append(dict(name=name, kind="fixed" if v is not None else "off", value=v)); continue
        symbols = cas.symvar(v)
        bounds = []
        for c, d in calls:
            if isinstance(c, list) or not c.is_op(cas.OP_LE):
                continue
            s = cas.symvar(c)
            if len(s) == 1 and len(symbols) == 1 and s[0].name() == symbols[0].name():
                left, right = c.dep(0), c.dep(1)      # c is "left <= right"
                var_on_right = cas.depends_on(right, symbols[0])
                bound_side = left if var_on_right else right
                var_side = right if var_on_right else left
                if cas.symvar(bound_side):
                    continue
                scale = float(sol.value(v)) / float(sol.value(var_side))
                bounds.append(dict(kind="lower" if var_on_right else "upper",
                                   bound=float(sol.value(bound_side)) * scale, scale=scale,
                                   dual=float(sol.value(d))))
        derived = len(symbols) != 1 or name in ("energy_capacity_battery_J", "power_max_discharge_battery_W")
        rows.append(dict(name=name, kind="derived" if derived else "variable", value=float(sol.value(v)),
                         bounds=bounds if not derived else []))
    opti = x.opti
    lbg, ubg = np.array(sol.value(opti.lbg)).reshape(-1), np.array(sol.value(opti.ubg)).reshape(-1)
    where = lambda declarations: [(str(path).split("size_halo/")[-1], line, n) for path, line, _, n
                                  in declarations.values()]
    out = dict(
        count_equalities=int(np.sum(lbg == ubg)), count_inequalities=int(np.sum(lbg != ubg)),
        variable_declarations=where(opti._variable_declarations),
        constraint_declarations=where(opti._constraint_declarations),
        nx=x.opti.nx, ng=x.opti.ng, iterations=sol.stats()["iter_count"], status=sol.stats()["return_status"],
        time_s=x.time_s, result=r, count_calls=len(calls),
        margins=[(m.label, float(sol.value(m.value)), float(sol.value(dual))) for m, dual in zip(margins, calls[index][1])],
        closure=[(str(c)[:60], float(sol.value(d))) for c, d in zip(closure_constraints, closure_duals)],
        variables=rows)
    return out

xray = cached("c13_xray", compute_xray)
print(f"{xray['status']}: {xray['iterations']} IPOPT iterations, {xray['time_s']:.0f} s inside IPOPT")
print(f"nx = {xray['nx']} variables, ng = {xray['ng']} constraints "
      f"({xray['count_equalities']} equalities, {xray['count_inequalities']} inequalities incl. bounds), "
      f"{xray['count_calls']} subject_to calls")
check("warm re-solve converges to the cached baseline MTOM (< 1e-3 kg)",
      abs(xray["result"].mass_takeoff_kg - ref.mass_takeoff_kg) < 1e-3)
check("equalities + inequalities = ng", xray["count_equalities"] + xray["count_inequalities"] == xray["ng"])

Solve_Succeeded: 88 IPOPT iterations, 88 s inside IPOPT
nx = 140 variables, ng = 964 constraints (80 equalities, 884 inequalities incl. bounds), 1017 subject_to calls
PASS  warm re-solve converges to the cached baseline MTOM (< 1e-3 kg)
PASS  equalities + inequalities = ng


**Where the 140 variables and 964 constraints are declared** (AeroSandbox records the file and line of every
`opti.variable` and `subject_to`):

In [32]:
from collections import Counter
for title, key, total in (("variables", "variable_declarations", xray["nx"]), ("constraints", "constraint_declarations", xray["ng"])):
    by_line = Counter()
    for path, line, n in xray[key]:
        by_line[(path, line)] += n
    print(f"{title}:")
    by_file = Counter()
    for (path, line), n in sorted(by_line.items()):
        by_file[path] += n
    for path, n in by_file.items():
        lines = [f"L{line}x{k}" if k > 1 else f"L{line}" for (p, line), k in sorted(by_line.items()) if p == path]
        print(f"  {n:4d} in {path}: {' '.join(lines)}")
    check(f"declared {title} add up to the problem size", sum(by_file.values()) == total)

variables:
    21 in examples/halo_sizing.py: L1093 L1101 L1102 L1103 L1104 L1105 L1106 L1109 L1112 L1119 L1125 L1127 L1128 L1132 L1136 L1141 L1148 L1150 L1156 L1177 L1179
   119 in src/aircraft_closure/performance/flight_point.py: L183x13 L189x10 L190x10 L195x10 L208x10 L237x23 L288x20 L291x23
PASS  declared variables add up to the problem size
constraints:
   573 in examples/halo_sizing.py: L1093 L1101x2 L1102x2 L1103 L1104 L1105 L1106 L1109 L1112 L1119 L1125 L1127x2 L1128 L1132x2 L1136x2 L1141x2 L1148x2 L1150x2 L1156x2 L1177x2 L1179x2 L1299x4 L1305x537
    54 in src/aircraft_closure/mission/mission.py: L114x18 L126x36
   337 in src/aircraft_closure/performance/flight_point.py: L183x13 L189x20 L190x10 L195x10 L202x20 L208x10 L211x10 L221x23 L223x23 L225x10 L241x23 L288x40 L291x23 L383x10 L388x69 L395x23
PASS  declared constraints add up to the problem size


Reading it:
- **21 variables** are declared in this file: MTOM (L1093), the 18 design variables of L1101-1158 that are on, and
  the two mission speeds (L1177, L1179). **119** come from `flight_point.py`, one set per point: rotor speed (L183
  hover, L189 airplane), angle of attack (L190), blown-wing thrust (L195), cooling drag (L208), battery current
  (L237), battery share (L288), generator torque (L291). There are **23 flight points**: 13 hovers (take-off and
  landing, 3 engine-out, 6 failure points, the hover requirement, the hot day) and 10 airplane-mode points (7
  mission sub-points: climb, 4 cruise, loiter, descent; plus the climb, max-speed and ceiling requirements). Battery
  current and generator torque exist at all 23; the battery share at 20, because the three airplane requirement
  points are flown on the turbogenerators only (share fixed at 0).
- **537 margins** are one `subject_to` call (L1305); the **4 closure equalities** one call (L1299); the bounds of
  the 21 local variables 32 rows (11 have both bounds); the rest are physics equalities and limits inside flight points (lift = weight,
  stall, blade loading, tip Mach, power balances) and the mission's SOC window (`mission.py` L126).

### 10.3 Every design variable at the solution

In [33]:
rows_table = []
for row in xray["variables"]:
    if row["kind"] in ("off",):
        print(f"  {row['name']:<32} off (None)"); continue
    if row["kind"] == "fixed":
        print(f"  {row['name']:<32} {row['value']:12.6g}  fixed (a number)"); continue
    if row["kind"] == "derived":
        print(f"  {row['name']:<32} {row['value']:12.6g}  derived from count_parallel_battery"); continue
    declared = []
    for kind in ("lower", "upper"):
        first = next((b for b in row["bounds"] if b["kind"] == kind), None)
        if first is not None:
            declared.append(first)
    status = []
    for b in declared:
        gap = abs(row["value"] - b["bound"]) / max(abs(b["bound"]), 1e-9)
        if gap < 1e-6 or b["dual"] > 1e-6:
            # multiplier of (x / s >= lb / s): d(obj)/d(lb / s). Tightening the bound by 1 % of its value moves
            # lb / s by 0.01 lb / s, i.e. MTOM by 1000 x dual x 0.01 x lb / s.
            kg = 1000 * b["dual"] * 0.01 * abs(b["bound"]) / b["scale"]
            status.append(f"ON {b['kind']} bound {b['bound']:.6g} (dual {b['dual']:.4f}, {kg:.2f} kg MTOM per 1 % tighter)")
            rows_table.append((row["name"], b["kind"], b["bound"], b["dual"], kg))
    bounds_text = ", ".join(f"{b['kind'][0]}={b['bound']:.4g}" for b in declared)
    print(f"  {row['name']:<32} {row['value']:12.6g}  scale {declared[0]['scale']:.4g}  [{bounds_text}]  "
          + ("; ".join(status) if status else "interior"))
active = {name: kind for name, kind, *_ in rows_table}
check("exactly two design variables sit on a bound: solidity (lower) and design tip speed (upper)",
      active == {"solidity": "lower", "speed_tip_m_s": "upper"})
check("tip-speed upper bound is 0.70 x sea-level speed of sound",
      abs(d.speed_tip_m_s - 0.70 * asb.Atmosphere(altitude=0).speed_of_sound()) < 1e-5)

  x_le_wing_m                           4.62351  scale 4.624  [l=3, u=8.5]  interior
  area_wing_m2                          23.2987  scale 10  [l=5, u=60]  interior
  area_horizontal_tail_m2                5.2463  scale 5.246  [l=0.5]  interior
  area_vertical_tail_m2                 2.73196  scale 2.732  [l=0.5]  interior
  torque_peak_motor_Nm                  1616.94  scale 1000  [l=100]  interior
  torque_peak_generator_Nm              891.323  scale 1000  [l=100]  interior
  power_rated_turboshaft_W               835184  fixed (a number)
  mass_turboshaft_bare_kg               184.207  scale 100  [l=10]  interior
  power_max_discharge_battery_W          850259  derived from count_parallel_battery
  energy_capacity_battery_J         2.51978e+08  derived from count_parallel_battery
  area_disk_m2                            73.36  scale 10  [l=5, u=120]  interior
  mass_fuel_kg                          936.443  scale 500  [l=0]  interior
  solidity                                 0.

Two design variables are on bounds, both rotor variables:
- **design tip speed at its Mach 0.70 limit** (238.2 m/s). Higher tip speed means a smaller, lighter rotor for the
  same thrust (higher speed lowers the blade loading CT/σ for a given solidity); the bound stops it.
- **solidity at its 0.06 lower bound**. Less blade area means less profile power and a lighter rotor; the
  blade-loading limit CT/σ <= 0.14 (`rotor.py` L72, enforced at `flight_point.py` L221) is met with equality at the
  hovers (`ref.hovers` shows 0.140). That limit is a plain `subject_to`, not a margin, so its multiplier is not in the
  margin table.

Every other variable is interior. The rotor gear ratio (4.53) and motor speed (204 rad/s, 1,950 rpm) sit well inside
their bounds: with whole Evolito units (2,500 rpm max) the slow-motor trade that plan 033 worried about does not push
anything to a limit.

In [34]:
for h in ref.hovers:
    print(f"  {h.label:<15} CT/sigma {h.blade_loading:.5f}")
check("blade loading is at its 0.14 limit at the hover requirement point", abs(ref.hovers[0].blade_loading - 0.14) < 1e-6)

  hover           CT/sigma 0.14000
  take-off hover  CT/sigma 0.14000
  landing hover   CT/sigma 0.14000
  hot-day hover   CT/sigma 0.14000
PASS  blade loading is at its 0.14 limit at the hover requirement point


### 10.4 Aircraft, mission, requirement points, engine out, hot day, failures

In [35]:
show_source("examples/halo_sizing.py", 1183, 1250)

# examples/halo_sizing.py  lines 1183-1250
1183      # ---- Aircraft, mission, requirements ----------------------------------------------
1184      if objective == "payload":
1185          mass_payload_kg = opti.variable(init_guess=initial.mass_payload_kg if initial else 600.0, scale=100.0,
1186                                          lower_bound=0.0)
1187          r = replace(r, mass_payload_kg=mass_payload_kg)
1188      aircraft = build_halo_aircraft(design, r, a, factors, lapse_exponent)
1189      instances = aircraft.powertrain.topology.instances
1190      battery = instances["battery"].component
1191      if is_ecm:
1192          design = replace(design, energy_capacity_battery_J=battery.energy_capacity_J,
1193                           power_max_discharge_battery_W=battery.power_max_discharge_W)
1194      thermal = a.thermal_model
1195      flown = build_mission(opti, aircraft, aerodynamics, mission, mass_takeoff_kg, soc_take_off,
1196                            hybridization_e

- **L1184-1187 payload objective**: payload becomes a variable (init 600 kg, scale 100, >= 0) and is written into
  the requirements with `replace`, so every item built from `r` (the payload item L893, the requirement set) sees the
  variable.
- **L1188** the aircraft is built **once**, symbolically, from the design dataclass of variables. **L1189-1193** with
  the ECM pack, energy and power are read back from the built pack and written into `design`, so the rest of the
  function (cost, SOC formulas) does not need to know which battery model is used.
- **L1195-1201 the mission** (C12): six segments chained by mass and SOC from MTOM and SOC 0.95, battery share free
  down to -1 (recharge), SOC floor 0.30 at every segment end, ECM subsegments `(1, 1, 4, 1, 1, 1)`, and with the
  thermal model the machines start at the coolant temperature and carry their temperature from point to point.
- **L1203-1207 requirement points** at MTOM: hover, climb, max speed, ceiling. With the thermal model the hover
  requirement is a 60 s hover from a cold start (coolant temperature) so the short-time rating applies; the airplane
  points are steady.
- **L1208-1212** handles to the take-off hover, cruise and loiter; L1210 the machine temperatures at the end of the
  take-off hover (last subsegment), where the engine fails.
- **L1213-1225 engine-out hover (ECM)**: a one-segment mission at MTOM with one turbogenerator inactive, from the SOC
  floor (0.30) to the emergency floor (0.10), split into 3 points so the voltage sag at low SOC is seen, polarization
  already developed (`"steady"`), thermal start from the end of the take-off hover. Its margins and its end SOC are
  collected.
- **L1226-1234** the same as a single flight point for the constant battery; the SOC drop is chemical energy over
  capacity.
- **L1235-1246 hot-day hover** (Tier 16): 4,000 ft at ISA + 27.7 K, T/W 1.05, at the **end-of-mission mass and SOC**
  (`flown.mass_end_kg`, `flown.soc_end`), thermal start from the end of the mission. L1245-1246 compute the SOC after
  60 s as chemical energy / pack energy, also with the ECM pack (whose own points count coulombs, see Findings).
- **L1247-1249 failure hovers** (Tier 18), one more coupled mission per case, from the end of the take-off hover's
  temperatures.

Every point is one more set of variables and equations on the **same** aircraft: the engine-out and failure hovers
size the machines and the battery in the same solve that minimizes MTOM. Nothing is checked after the fact.

In [36]:
labels = [label for label, _, _ in xray["margins"]]
groups = Counter(label.split(":")[0].split(" (")[0] for label in labels)
for group, n in groups.items():
    print(f"  {n:3d}  {group}")
check("three engine-out points (subsegments_engine_out = 3)", sum(g.startswith("engine-out hover ") for g in groups) == 3)
check("failure cases in the problem: bus out and string out, 3 points each (lane out is covered by bus out, L1410)",
      sorted(g for g in groups if g.startswith(("bus-out", "string-out", "lane-out")))
      == [f"{c} {i}/3" for c in ("bus-out hover", "string-out hover") for i in (1, 2, 3)])

    1  static_margin
    1  cn_beta_per_rad
    1  soc_end
    1  soc_after_engine_out_hover
    1  stall_lift_at_120kt
    1  rotor_radius_m
   22  take-off hover
   43  climb
   22  cruise 1/4
   22  cruise 2/4
   22  cruise 3/4
   22  cruise 4/4
   22  reserve loiter
   22  descent
   22  landing hover
   22  engine-out hover 1/3
   22  engine-out hover 2/3
   22  engine-out hover 3/3
   21  hover
   21  max_speed
   21  ceiling
   25  bus-out hover 1/3
   25  bus-out hover 2/3
   25  bus-out hover 3/3
   22  string-out hover 1/3
   22  string-out hover 2/3
   22  string-out hover 3/3
   10  whirl flutter torsion per rev
   10  whirl flutter beam per rev
   21  hot-day hover
    1  soc_after_hot_day_hover
PASS  three engine-out points (subsegments_engine_out = 3)
PASS  failure cases in the problem: bus out and string out, 3 points each (lane out is covered by bus out, L1410)


### 10.5 Mass closure, stability, design margins, whirl flutter

In [37]:
show_source("examples/halo_sizing.py", 1251, 1298)

# examples/halo_sizing.py  lines 1251-1298
1251      # ---- Mass closure ------------------------------------------------------------------
1252      condition = StructuralDesignCondition(mass_design_kg=mass_takeoff_kg, load_factor_ultimate=a.load_factor_ultimate,
1253                                            velocity_cruise_m_s=cruise.segment.velocity_m_s,
1254                                            altitude_cruise_m=altitude_cruise_m,
1255                                            lift_to_drag_cruise=cruise.point.aero.lift_to_drag)
1256      breakdown = aircraft.get_mass_breakdown(condition)
1257      total = breakdown.total()
1258      x_cg_m = total.x_cg
1259  
1260      # ---- Stability, geometry, stall -----------------------------------------------------
1261      static_margin = longitudinal.static_margin(aircraft, aerodynamics, x_cg_m, 80.0, 1000.0)
1262      cn_beta_per_rad = directional.cn_beta_per_rad(aircraft, aerodynamics, x_cg_m, 80.0, 1000.0)
1263      radius_m =

- **L1252-1256 structural condition**: design mass = MTOM (a variable), n_ult 4.5, and the **solved** cruise speed,
  altitude and L/D. The Raymer correlations use cruise dynamic pressure and L/D; feeding them the variables keeps the
  structure coupled to the mission without an outer loop (decision 010). `get_mass_breakdown` sums all items
  (`asb.MassProperties`), **L1257-1258** total mass and CG.
- **L1261-1262** static margin and Cn_beta at 80 m/s and 1,000 m (C10), about the solved CG.
- **L1263-1267** rotor radius, span, sea-level density, stall lift at 120 kt with CL_max 1.5, weight at MTOM.
- **L1269-1276 design margins**, each normalized by its limit (`core/margins.py`, C01):

$$m_{\mathrm{above}} = \frac{v - \ell}{\ell}, \qquad m_{\mathrm{below}} = \frac{\ell - v}{\ell}$$

  - static margin >= 0.10 and Cn_beta >= 0.06 /rad (decision 007);
  - mission end SOC >= the reserve SOC, SOC after the engine-out hover >= 0.10;
  - stall lift at 120 kt >= weight;
  - rotor radius <= (span - fuselage width) / 2 - 0.3 m: the rotors may not reach the fuselage. With tip-mounted
    rotors this couples disk area to wing span, i.e. to wing area and aspect ratio.
- **L1277-1279** all margins: design, mission, engine out, requirement points, failure hovers.
- **L1280-1293 whirl flutter** (Tier 20): the wing's realized torsion and beam frequencies (from the AFDD wing built
  for the design rotor speed) divided by each airplane-mode point's rotor speed must stay at or above the XV-15
  placement (1.09 and 0.43 per rev). 10 airplane points (7 mission sub-points, climb, max speed, ceiling) x 2 = 20
  margins. This is how the design rotor speed variable is closed: a wing built for a higher speed is heavier, a
  lower one fails the margin at the fastest-turning point.
- **L1294-1297 hot-day margins**, plus SOC after the hot-day hover >= 0.10 (L1295: an alternative contingency to the
  engine-out reserve, starting from the end SOC instead of the 0.30 floor).

In [38]:
margins = {label: (value, dual) for label, value, dual in xray["margins"]}
for label in ("static_margin", "cn_beta_per_rad", "soc_end", "soc_after_engine_out_hover", "stall_lift_at_120kt",
              "rotor_radius_m", "soc_after_hot_day_hover"):
    print(f"  {label:<28} margin {margins[label][0]:+.4f}")
check("static margin and Cn_beta sit on their limits (0.10, 0.06)",
      abs(ref.static_margin - 0.10) < 1e-6 and abs(ref.cn_beta_per_rad - 0.06) < 1e-6)
limit_m = (ref.span_m - a.diameter_fuselage_m) / 2 - a.clearance_rotor_fuselage_m
print(f"rotor radius {ref.radius_rotor_m:.4f} m, limit (span {ref.span_m:.4f} - width {a.diameter_fuselage_m:.4f}) / 2 - 0.3 = {limit_m:.4f} m")
check("rotor radius sits on its span limit (L1275)", abs(ref.radius_rotor_m - limit_m) < 1e-6)
check("20 whirl-flutter margins: 10 airplane-mode points x (torsion, beam)",
      sum(label.startswith("whirl flutter") for label in labels) == 20 and len(ref.whirl_flutter) == 10)
check("total margins = 537", len(labels) == 537)

  static_margin                margin -0.0000
  cn_beta_per_rad              margin -0.0000
  soc_end                      margin +0.4485
  soc_after_engine_out_hover   margin +0.0589
  stall_lift_at_120kt          margin +0.2082
  rotor_radius_m               margin -0.0000
  soc_after_hot_day_hover      margin +2.5366
PASS  static margin and Cn_beta sit on their limits (0.10, 0.06)
rotor radius 4.8323 m, limit (span 11.9410 - width 1.6764) / 2 - 0.3 = 4.8323 m
PASS  rotor radius sits on its span limit (L1275)
PASS  20 whirl-flutter margins: 10 airplane-mode points x (torsion, beam)
PASS  total margins = 537


### 10.6 Closure equalities, margins, cost, objective

In [39]:
show_source("examples/halo_sizing.py", 1299, 1330)

# examples/halo_sizing.py  lines 1299-1330
1299      opti.subject_to([
1300          mass_takeoff_kg / total.mass == 1,                                           # mass closure
1301          x_cg_m == aircraft.wing.x_le_root_m + 0.25 * aircraft.wing.chord_root_m(),   # hover pitch trim
1302          design.mass_fuel_kg == fuel_factor * flown.mass_fuel_burnt_kg,               # fuel with reserve
1303          power_turboshaft(design.mass_turboshaft_bare_kg) / design.power_rated_turboshaft_W == 1,  # engine regression
1304      ])
1305      opti.subject_to([m.value >= 0 for m in all_margins])
1306      # ---- Cost per mission (Tier 22) -----------------------------------------------------------
1307      motor_component, generator_component = instances["motor"].component, instances["generator"].component
1308      turboshaft_component = instances["turboshaft"].component
1309      mass_airframe_kg = (breakdown.mass_empty_kg() - battery.get_mass()
1310                          - a.count_ro

**L1299-1304, the four closure equalities.** These are what make the problem a *sizing* problem:

1. **L1300 mass closure**, written as a ratio so the residual is order one:

$$\frac{m_{TO}}{\sum_i m_i(\text{design}, m_{TO})} = 1$$

   The parts depend on MTOM (structure through the design condition, battery and machines through the points), so
   this one equality replaces the classic fixed-point iteration on MTOM.
2. **L1301 hover pitch trim**: the CG at the wing quarter chord, where the rotors are (L824). It is what sets
   `x_le_wing_m`.
3. **L1302 fuel**: fuel loaded = 1.1 x fuel burnt over the mission (`fuel_factor`).
4. **L1303 engine regression**: the AeroSandbox mass-power regression must give the fixed rating. With fixed engines
   it pins the bare mass (184.2 kg) and so the thermal efficiency, which depends on mass.

**L1305** all 537 margins `>= 0` in one call (one list, so one `duals` list).

**L1306-1323 cost per mission** (Tier 22, C12): airframe mass = empty mass minus battery, motors, generators and
turboshafts; the battery's throughput is a smooth `max(E, 0)` per point (charging is not throughput) with a
softness of 0.1 % of capacity; the ground recharge energy returns the pack to SOC 0.95. L1310 and L1316 multiply the
motor by `count_rotors` only, but there are `count_rotors x count_lanes_motor` = 4 lane motors (C12 Findings). The
cost is an expression in every case; it is minimized only with `objective="cost"`.

**L1324-1329 objective**, each scaled to order one to ten: `-payload / 100`, `cost / 100 USD`, or `MTOM / 1000`.

**Sign conventions of the multipliers (verify them before reading the numbers).** The toy problems below are
solved with the same `asb.Opti` API:
- an inequality written `expr >= c` returns a dual μ >= 0 with μ = d(obj*)/d(c): raising the right-hand side by ε
  raises the optimum objective by μ ε. The margins are `m >= 0`, so tightening a margin by 1 % (requiring
  m >= 0.01) raises MTOM by 1000 x μ x 0.01 = 10 μ kg (the objective is MTOM / 1000);
- an equality `A == B` returns ν = -d(obj*)/d(e), where the constraint is perturbed to `A - B = e` (CasADi keeps a
  constant on the right whatever side it was written on). So **a negative equality dual means that increasing
  A - B costs objective**.

In [40]:
def toy(form):
    opti = asb.Opti(); x = opti.variable(init_guess=1.0); y = opti.variable(init_guess=1.0)
    dual = opti.subject_to(eval(form)); opti.minimize(x**2 + 3 * (y - 1)**2)
    solution = opti.solve(verbose=False)
    return solution.value(dual), solution.value(x)
# min x^2 + 3 (y - 1)^2. With x = y + 2 + e: d(obj*)/de = 2 x* = 4.5 (x* = 2.25).
for form in ("x >= y + 2", "x == y + 2", "y + 2 == x", "x / (y + 2) == 1"):
    dual, x_value = toy(form)
    print(f"  {form:<18} dual {dual:+.4f}  (x* = {x_value:.4f})")
check("inequality A >= B: dual = +d(obj*)/d(shift of B) = +4.5", abs(toy("x >= y + 2")[0] - 4.5) < 1e-6)
check("equality A == B: dual = -d(obj*)/d(A - B) = -4.5; written B == A it flips to +4.5",
      abs(toy("x == y + 2")[0] + 4.5) < 1e-6 and abs(toy("y + 2 == x")[0] - 4.5) < 1e-6)

  x >= y + 2         dual +4.5000  (x* = 2.2500)


  x == y + 2         dual -4.5000  (x* = 2.2500)


  y + 2 == x         dual +4.5000  (x* = 2.2500)


  x / (y + 2) == 1   dual -10.1250  (x* = 2.2500)
PASS  inequality A >= B: dual = +d(obj*)/d(shift of B) = +4.5


PASS  equality A == B: dual = -d(obj*)/d(A - B) = -4.5; written B == A it flips to +4.5


In [41]:
names = ["mass closure  m_TO / sum(m) == 1", "CG trim  x_cg == x_le + c/4", "fuel  m_fuel == 1.1 m_burnt",
         "engine  P(m_bare) / P_rated == 1"]
closure = [dual for _, dual in xray["closure"]]
for name, dual in zip(names, closure):
    print(f"  {name:<36} dual {dual:+.6f}  ->  d(MTOM)/d(e) = {-1000 * dual:+.2f} kg per unit of A - B")
growth = -1000 * closure[0] / ref.mass_takeoff_kg
print(f"mass closure: requiring m_TO = (1 + e) sum(m) adds {-1000 * closure[0] * 0.01:.1f} kg per e = 1 %; "
      f"per kg of extra mass that is {growth:.3f} kg of MTOM")
print(f"fuel: 1 kg of fuel above the 1.1 x burnt rule costs {-1000 * closure[2]:.3f} kg of MTOM")
print(f"CG: moving the CG 1 cm aft of the rotor axis costs {-1000 * closure[1] * 0.01:.2f} kg; "
      f"engine: a 1 % heavier regression (P(m) = 1.01 P_rated) costs {-1000 * closure[3] * 0.01:.2f} kg")
check("all four closure duals are negative: each A - B costs MTOM when increased", all(dual < 0 for dual in closure))
check("growth factor from mass closure equals the fuel dual (both: 1 kg of dead mass costs the same MTOM)",
      abs(growth - (-1000 * closure[2])) / growth < 1e-3)

  mass closure  m_TO / sum(m) == 1     dual -17.466589  ->  d(MTOM)/d(e) = +17466.59 kg per unit of A - B
  CG trim  x_cg == x_le + c/4          dual -0.212566  ->  d(MTOM)/d(e) = +212.57 kg per unit of A - B
  fuel  m_fuel == 1.1 m_burnt          dual -0.002537  ->  d(MTOM)/d(e) = +2.54 kg per unit of A - B
  engine  P(m_bare) / P_rated == 1     dual -0.876762  ->  d(MTOM)/d(e) = +876.76 kg per unit of A - B
mass closure: requiring m_TO = (1 + e) sum(m) adds 174.7 kg per e = 1 %; per kg of extra mass that is 2.537 kg of MTOM
fuel: 1 kg of fuel above the 1.1 x burnt rule costs 2.537 kg of MTOM
CG: moving the CG 1 cm aft of the rotor axis costs 2.13 kg; engine: a 1 % heavier regression (P(m) = 1.01 P_rated) costs 8.77 kg
PASS  all four closure duals are negative: each A - B costs MTOM when increased
PASS  growth factor from mass closure equals the fuel dual (both: 1 kg of dead mass costs the same MTOM)


**Reading the four equality duals.**
- **Mass closure (-17.47).** Perturb to m_TO = (1 + e) Σm. Then MTOM rises by 17,466 e kg: 175 kg for e = 1 %.
  Divided by MTOM this is **2.54 kg of MTOM per kg of added dead mass**, the classic mass growth factor (snowball), here
  read from one multiplier.
- **Fuel (-0.00254).** Perturb to m_fuel = 1.1 m_burnt + e: 1 kg of extra fuel costs 2.54 kg of MTOM. The same 2.54,
  as it must be: one kg of fuel that is never burnt is dead mass. The check above confirms the two agree to 1e-3.
- **CG trim (-0.213 per m).** Perturb to x_cg = x_ac + e: putting the CG 1 cm aft of the rotor axis costs 2.1 kg (the
  tails and wing must re-trim around it). Small: the wing position absorbs it.
- **Engine regression (-0.877).** Perturb to P(m)/P_rated = 1 + e: a regression 1 % more pessimistic about
  power per mass (each bare engine about 1 % / 1.04 = 0.96 % heavier, `dlnP/dlnm = 1.04`) costs 8.8 kg of MTOM.
  A pure dead-mass estimate (2 engines x 1.768 kg x 1.145 powerplant factor x 2.54 = 10.3 kg) is higher. A likely
  reason, not separated here: the efficiency regression also rises with bare mass (d ln η / d ln m = 0.16), so the
  heavier engine burns less fuel.

### 10.7 The solve and the result extraction

In [42]:
show_source("examples/halo_sizing.py", 1331, 1401)

# examples/halo_sizing.py  lines 1331-1401
1331      solution = opti.solve(verbose=verbose, max_iter=max_iter)
1332      value = lambda expression: float(solution.value(expression))
1333      report = margin_report(all_margins, solution.value)
1334      altitude_cruise_value_m = value(altitude_cruise_m)
1335      altitudes = [(0, 0), (0, altitude_cruise_value_m), (altitude_cruise_value_m,) * 2, (altitude_cruise_value_m,) * 2,
1336                   (altitude_cruise_value_m, 0), (0, 0)]
1337      return HaloSizingResult(
1338          mass_takeoff_kg=value(mass_takeoff_kg), mass_payload_kg=value(r.mass_payload_kg),
1339          mass_empty_kg=value(breakdown.mass_empty_kg()),
1340          mass_fuel_kg=value(design.mass_fuel_kg), mass_fuel_burnt_kg=value(flown.mass_fuel_burnt_kg),
1341          design=HaloDesign(**{f.name: value(getattr(design, f.name)) if getattr(design, f.name) is not None else None
1342                               for f in fields(HaloDesign)}),
1343          energy

- **L1331** the one IPOPT call. `max_iter` 3,000. A failure raises `RuntimeError` (caught by the dispatcher and the
  multistart).
- **L1332** `value` turns any expression (or number) into a float through `solution.value`.
- **L1333** the margin report sorted from most critical (C01).
- **L1334-1336** the segment altitudes for the table (start, end), numeric.
- **L1337-1399 `HaloSizingResult`**, everything numeric:
  - L1341-1342 the design rebuilt field by field (`None` stays `None`): this is the warm start of any later solve;
  - L1345-1346 disk loading on the total disk area of both rotors, wing loading;
  - L1347-1348 motor and generator ratings: **per unit instance** (the motor is one lane);
  - L1352 closure residual in kg (zero to solver tolerance), L1353-1354 the regression residual;
  - L1356-1369 one dict per mission segment: the reported point is the segment's first sub-point (C12 Findings);
  - L1370 `binding`: margins with |value| < 1e-4; L1371 the smallest margin;
  - L1372-1374 component and powertrain masses; L1375-1378 hover summaries; L1380-1382 battery traces;
    L1383-1388 wing breakdown and whirl margins per point; L1389 the cost; L1390 the unit counts; L1391-1396 thermal
    trace; L1397-1398 lanes and failure summaries.

### 10.8 The x-ray: margin multipliers, ranked

For each margin `m >= 0` the dual μ is d(MTOM/1000)/d(rhs). The column "kg / 1 %" is the MTOM increase if that
margin were required to be at least +1 % instead of 0: 1000 x μ x 0.01.

In [43]:
ranked = sorted(xray["margins"], key=lambda row: -row[2])
print(f"{'dual':>9} {'kg/1%':>7} {'margin':>10}  label")
for label, value, dual in ranked:
    if dual > 1e-6:
        print(f"{dual:9.4f} {1000 * dual * 0.01:7.2f} {value:+10.2e}  {label}")
positive = [row for row in xray["margins"] if row[2] > 1e-6]
print(f"{len(positive)} margins with a non-zero multiplier out of {len(xray['margins'])}")
check("every margin multiplier is non-negative (to 1e-8)", min(row[2] for row in xray["margins"]) > -1e-8)
check("complementarity: every margin with a multiplier > 1e-4 sits at zero (|m| < 1e-6)",
      all(abs(value) < 1e-6 for _, value, dual in xray["margins"] if dual > 1e-4))
check("the margins with a multiplier are exactly the result's binding set",
      {row[0] for row in positive} == set(ref.binding) == set(xray["result"].binding))

     dual   kg/1%     margin  label
   1.3205   13.21  -9.99e-09  rotor_radius_m
   1.1683   11.68  -9.99e-09  engine-out hover 3/3: battery end voltage_V
   0.3147    3.15  -9.97e-09  hot-day hover: heat_exchanger power_heat_W
   0.3006    3.01  -9.97e-09  engine-out hover 3/3: generator_gearbox power_input_W
   0.3006    3.01  -9.97e-09  engine-out hover 3/3: turboshaft power_shaft_W
   0.2830    2.83  -9.96e-09  hover: propulsor power_shaft_W
   0.2830    2.83  -9.96e-09  hover: gearbox power_input_W
   0.2420    2.42  -9.96e-09  engine-out hover 1/3: turboshaft power_shaft_W
   0.2420    2.42  -9.96e-09  engine-out hover 1/3: generator_gearbox power_input_W
   0.2165    2.17  -9.95e-09  hot-day hover: turboshaft power_shaft_W
   0.1715    1.72  -9.94e-09  engine-out hover 2/3: turboshaft power_shaft_W
   0.1715    1.72  -9.94e-09  engine-out hover 2/3: generator_gearbox power_input_W
   0.1022    1.02  -9.90e-09  static_margin
   0.0378    0.38  -9.75e-09  bus-out hover 1/3: motor 

**What sizes the Halo** (from the table, ranked by cost):

1. **Rotor-to-fuselage span clearance (13.2 kg per 1 %).** The rotors want more disk (lower hover power) than the
   wing span allows. The only ways out are a larger span (a heavier wing) or a smaller clearance. Section 10.9 checks
   this number by finite difference.
2. **Engine-out hover, last point: battery end voltage (11.7 kg per 1 %).** At the end of the 60 s engine-out hover
   (SOC near 0.10) the pack voltage reaches the cell minimum. This sizes the parallel string count.
3. **Hot-day heat rejection (3.1 kg per 1 %)**: the heat exchanger rating is set by the hot-day hover.
4. **Turboshaft power at the engine-out and hot-day hovers, and rotor/gearbox power at the hover requirement
   (1.7-3.0 kg per 1 % each).** The fixed 1,120 hp engines are at their available power in all three engine-out points
   and in the hot-day hover; the generator step-up gearbox rating binds with them (the same power, so the same
   multiplier, split between two identical constraints). The rotor shaft power and gearbox input power at the
   4,000 ft hover bind together for the same reason.
5. **Static margin and Cn_beta (1.0 and 0.4 kg per 1 %)**: tail sizing.
6. **Bus-out lane torque (0.4 kg)** and generator torque in the engine-out hover (under 0.1 kg).

Pairs with equal multipliers (turboshaft and generator-gearbox power, rotor and gearbox power) are **degenerate**:
two constraints that are the same function at the solution. IPOPT splits the multiplier between them; only their sum
is meaningful.

### 10.9 Finite-difference check of one multiplier

The rotor-radius margin is

$$m = \frac{\ell - R}{\ell}, \qquad \ell = \frac{b - w}{2} - c$$

with clearance c = 0.3 m. Raising the clearance by δ = 0.05 m gives the constraint R <= ℓ - δ, which is
m >= δ / ℓ with ℓ at the optimum (equal to R there, 4.832 m): a margin shift of 0.05 / 4.832 = 1.03 %. Predicted
MTOM change: 1000 x μ x δ / ℓ. The re-solve below (warm, cached) gives the actual change.

In [44]:
def compute_fd():
    ref, a = baseline(), baseline_assumptions()
    r = solve_halo_sizing_once(HaloRequirements(), replace(a, clearance_rotor_fuselage_m=0.35), initial=ref)
    return dict(mass_takeoff_kg=r.mass_takeoff_kg, radius_rotor_m=r.radius_rotor_m, span_m=r.span_m,
                binding=r.binding)

fd = cached("c13_fd", compute_fd)
mu = margins["rotor_radius_m"][1]
shift = 0.05 / limit_m
predicted_kg = 1000 * mu * shift
actual_kg = fd["mass_takeoff_kg"] - ref.mass_takeoff_kg
print(f"multiplier {mu:.4f}, margin shift {100 * shift:.3f} %: predicted +{predicted_kg:.2f} kg, re-solve +{actual_kg:.2f} kg "
      f"(ratio {actual_kg / predicted_kg:.4f})")
print(f"clearance 0.35 m: radius {fd['radius_rotor_m']:.4f} m (was {ref.radius_rotor_m:.4f}), span {fd['span_m']:.4f} m (was {ref.span_m:.4f})")
check("finite difference agrees with the multiplier prediction within 3 %", abs(actual_kg / predicted_kg - 1) < 0.03)
check("same binding set after the perturbation (the active set did not change)", set(fd["binding"]) == set(ref.binding))

multiplier 1.3205, margin shift 1.035 %: predicted +13.66 kg, re-solve +13.87 kg (ratio 1.0155)
clearance 0.35 m: radius 4.8386 m (was 4.8323), span 12.0536 m (was 11.9410)
PASS  finite difference agrees with the multiplier prediction within 3 %
PASS  same binding set after the perturbation (the active set did not change)


The re-solve is 1.5 % above the linear prediction. The residual is second order: the active set is unchanged, and
the multiplier itself grows as the margin tightens. So a multiplier is a reliable local price of a constraint. It is
not a forecast for large changes, which may change the active set.

---
## 11. Failure cases and the numeric summaries

In [45]:
show_source("examples/halo_sizing.py", 1402, 1472)

# examples/halo_sizing.py  lines 1402-1472
1402  def failure_hover_cases(assumptions):
1403      """Tier 18: ((label, HoverSegment counts), ...) for the failure hovers switched on; a case with nothing to
1404      fail onto (e.g. a lane out with one lane per rotor) raises."""
1405      a = assumptions
1406      lanes, engines = a.count_lanes_motor, a.count_turbogenerators
1407      cases = []
1408      # With one lane per rotor on each bus, a bus out is the lane-out state plus the tie (more demand, more heat), so
1409      # it covers the lane out; the duplicate point would only repeat its binding constraints (degenerate for IPOPT).
1410      lane_out_covered = a.failure_bus_out and a.count_buses > 1 and lanes // a.count_buses == 1
1411      if a.failure_lane_out and not (lane_out_covered and lanes > 1):
1412          cases.append(("lane-out hover", dict(active_lane_count=lanes - 1)))
1413      if a.failure_bus_out:
1414          cases.append(("bus-out hover", dict(count_buses_failed=1

- **L1402-1429 `failure_hover_cases`** turns the five flags into `(label, counts)` pairs for `HoverSegment`:
  - **L1410-1412**: with one lane per rotor on each bus (`lanes // buses == 1`), a bus out loses exactly one lane per
    rotor and also feeds through the tie, so it is the lane-out state plus more demand and heat. The lane-out point
    would repeat the bus-out's binding constraints (degenerate for IPOPT), so it is **skipped** even though
    `failure_lane_out=True`. In the baseline (2 lanes, 2 buses) the problem has bus out and string out only;
  - L1413-1422 one entry per remaining flag, with the reduced counts;
  - **L1423-1428** a case with nothing to fail onto (lane out with 1 lane, bus out with 1 bus, string out with 1
    string) raises `ValueError`, before any solve (L1035).
- **L1432-1448 `halo_failure_hovers`**: each case is a one-segment mission at MTOM from SOC 0.30 (`soc_minimum`, not
  the reserve variable) to the emergency floor, 3 ECM points, steady polarization, thermal start from the end of the
  take-off hover: the same set-up as the engine-out hover.
- **L1451-1470 `failure_summary`**: per case, the active counts, the worst lane torque / max torque, motor end
  temperature, battery and tie currents and the case's binding margins.

In [46]:
print("baseline:", [label for label, _ in failure_hover_cases(a)])
print("4 lanes, 2 buses:", [label for label, _ in failure_hover_cases(replace(a, count_lanes_motor=4))])
try:
    failure_hover_cases(replace(a, count_buses=1))
    raised = False
except ValueError as error:
    raised = True
    print("1 bus with failure_bus_out:", error)
check("lane out is dropped when each bus carries one lane per rotor (L1410)",
      [l for l, _ in failure_hover_cases(a)] == ["bus-out hover", "string-out hover"])
check("with 4 lanes on 2 buses (2 per bus) the lane-out case comes back",
      "lane-out hover" in [l for l, _ in failure_hover_cases(replace(a, count_lanes_motor=4))])
check("with 3 lanes on 2 buses, 3 // 2 = 1: lane out is still dropped",
      "lane-out hover" not in [l for l, _ in failure_hover_cases(replace(a, count_lanes_motor=3))])
check("an impossible failure case raises ValueError", raised)
for case in ref.failure_cases:
    print(f"  {case['label']:<17} SOC end {case['soc_end']:.3f}  lane torque/max {case['torque_lane_to_max']:.3f}  "
          f"tie {case['current_tie_A']:.0f} A  motor {case['temperature_end_motor_C']:.1f} C  binding {case['binding']}")
check("bus out: the surviving lane is at its unit max torque", abs(ref.failure_cases[0]["torque_lane_to_max"] - 1) < 1e-6)

baseline: ['bus-out hover', 'string-out hover']
4 lanes, 2 buses: ['lane-out hover', 'bus-out hover', 'string-out hover']
1 bus with failure_bus_out: The 'bus-out hover' case needs more than one unit to fail onto; set it False or raise the count.
PASS  lane out is dropped when each bus carries one lane per rotor (L1410)
PASS  with 4 lanes on 2 buses (2 per bus) the lane-out case comes back
PASS  with 3 lanes on 2 buses, 3 // 2 = 1: lane out is still dropped
PASS  an impossible failure case raises ValueError
  bus-out hover     SOC end 0.208  lane torque/max 1.000  tie 1065 A  motor 100.9 C  binding ('bus-out hover 1/3: motor torque_Nm',)
  string-out hover  SOC end 0.209  lane torque/max 0.524  tie 0 A  motor 83.4 C  binding ()
PASS  bus out: the surviving lane is at its unit max torque


In [47]:
show_source("examples/halo_sizing.py", 1473, 1576)

# examples/halo_sizing.py  lines 1473-1576
1473  @dataclass(frozen=True)
1474  class HoverSummary:
1475      """A solved hover point. `battery_share_min` is the battery share needed with the turbines at full power
1476      available (the turbine-limited share, generators at the point's efficiency); the solved `hybridization` may
1477      be higher wherever the split is free and not binding."""
1478      label: str
1479      altitude_m: float
1480      temperature_offset_K: float
1481      mass_kg: float
1482      soc: float
1483      hybridization: float
1484      battery_share_min: float
1485      power_rotors_W: float
1486      power_available_turboshafts_W: float
1487      power_battery_W: float
1488      blade_loading: Any = None             # CT / sigma (momentum + profile rotor only)
1489  
1490  
1491  def hover_summary(point, value, instances, assumptions):
1492      c = point.condition
1493      atmosphere = asb.Atmosphere(altitude=c.altitude_m, temperature_deviation=c.tempe

- **L1473-1488 `HoverSummary`** and **L1491-1511 `hover_summary`**: per hover point, the turbines' available power at
  that point's atmosphere and active engine count (L1493-1495), the generator efficiency at the point (L1496-1498),
  and the **minimum battery share** if the turbines ran at full available power (L1508):

$$h_{\min} = 1 - \frac{P_{\mathrm{avail}}\,\eta_{\mathrm{gen}}}{P_{\mathrm{el,motors}}}$$

  The solved share may be larger where the split is free (L1475-1477). Blade loading CT/σ at the point (L1501-1503)
  for the momentum + profile rotor only.
- **L1514-1521 `electrical_summary`**: bus voltage and layer losses (Tier 15 only).
- **L1524-1541 `thermal_trace`**: per point the heat by source, heat rejected, cooling drag, fan power, coolant flows
  and end temperatures (Tier 19).
- **L1544-1561 `heat_exchanger_summary`**: exchanger rating and mass, and per machine its rating, mass, continuous loss
  and thermal time constant (capacity x resistance).
- **L1564-1574 `battery_trace`**: per point the start time, SOC in and out, bus, end and open-circuit voltages, current
  and terminal power.

In [48]:
for h in ref.hovers:
    print(f"  {h.label:<15} {h.altitude_m / u.foot:5.0f} ft ISA{h.temperature_offset_K:+5.1f} K  {h.mass_kg:7.1f} kg  SOC {h.soc:.3f}  "
          f"share {h.hybridization:+.3f} (min {h.battery_share_min:+.3f})  rotors {h.power_rotors_W / 1e3:6.0f} kW  "
          f"turbines avail {h.power_available_turboshafts_W / 1e3:6.0f} kW")
check("every hover's solved battery share covers its turbine-limited minimum",
      all(h.hybridization >= h.battery_share_min - 1e-6 for h in ref.hovers))
hx = ref.heat_exchanger
print(f"heat exchanger {hx['power_rated_W'] / 1e3:.1f} kW, {hx['mass_kg']:.1f} kg; time constants "
      + ", ".join(f"{k} {v['time_constant_s']:.0f} s" for k, v in hx["machines"].items()))
check("exchanger mass = rating / 1 kW/kg", abs(hx["mass_kg"] - hx["power_rated_W"] / 1000) < 1e-9)
last = ref.engine_out_trace[-1]
print(f"engine-out last point: SOC {last['soc_start']:.3f} -> {last['soc_end']:.3f}, end voltage {last['voltage_end_V']:.1f} V")
check("engine-out hover ends at the pack minimum voltage (210 x 2.5 V = 525 V): the 2nd most expensive margin",
      abs(last["voltage_end_V"] - 525.0) < 1e-3)

  hover            4000 ft ISA +0.0 K   6885.1 kg  SOC 0.800  share +0.365 (min +0.097)  rotors   1485 kW  turbines avail   1518 kW
  take-off hover      0 ft ISA +0.0 K   6885.1 kg  SOC 0.950  share +0.072 (min -0.134)  rotors   1300 kW  turbines avail   1670 kW
  landing hover       0 ft ISA +0.0 K   6040.0 kg  SOC 0.466  share +0.113 (min -0.382)  rotors   1068 kW  turbines avail   1670 kW
  hot-day hover    4000 ft ISA+27.7 K   6033.8 kg  SOC 0.435  share +0.232 (min +0.229)  rotors   1277 kW  turbines avail   1115 kW
PASS  every hover's solved battery share covers its turbine-limited minimum
heat exchanger 126.9 kW, 126.9 kg; time constants motor 196 s, generator 136 s, battery 114 s
PASS  exchanger mass = rating / 1 kW/kg
engine-out last point: SOC 0.175 -> 0.106, end voltage 525.0 V
PASS  engine-out hover ends at the pack minimum voltage (210 x 2.5 V = 525 V): the 2nd most expensive margin


---
## 12. The multistart strategy (Tier 22, decision 029)

In [49]:
show_source("examples/halo_sizing.py", 1577, 1624)

# examples/halo_sizing.py  lines 1577-1624
1577  # ---- Tier 22 (plan 029): starting-point strategy ---------------------------------------------------------------
1578  # Every candidate start is one independent coupled solve of the target problem from an initial guess; a candidate
1579  # may first solve a simpler precursor problem (other aerodynamics, other battery, lower payload) to obtain that
1580  # guess. Nothing iterates to a fixed point: the list is finite and ordered, and the record says what happened.
1581  start_labels = ("caller", "redundancy_off", "strings_off", "electrical_off", "thermal_off", "mass_objective", "scholz_aero", "constant_battery",
1582                  "payload_continuation", "perturbed_low", "perturbed_high", "generic")
1583  fraction_payload_continuation = 0.85      # plan 026
1584  factor_perturbation_low = 0.85
1585  factor_perturbation_high = 1.15
1586  
1587  
1588  @dataclass(frozen=True)
1589  class StartAttempt:
1590      """One candidate start: 

- **L1578-1580** the principle: each candidate start is one independent coupled solve of the *target* problem; a
  candidate may first solve a simpler precursor problem to get its initial guess. Nothing iterates to a fixed point.
- **L1581-1582 `start_labels`**: the 12 known labels, used to validate a caller's list (L1729-1731).
- **L1583** payload continuation at 85 % (plan 026). **L1584-1585** perturbation factors 0.85 and 1.15.
- **L1588-1596 `StartAttempt`**: per candidate, success, MTOM, objective value, wall time and the last line of the
  IPOPT error. **L1599-1611 `StartRecord`**: the label that was returned plus every attempt; `spread_mass_takeoff_kg`
  is the local-optimum spread when `select="best"` ran several starts.
- **L1614-1622 `objective_value`**: the minimized quantity on a numeric result (payload negated so lower is better).

In [50]:
show_source("examples/halo_sizing.py", 1625, 1691)

# examples/halo_sizing.py  lines 1625-1691
1625  def default_starts(requirements, assumptions, objective):
1626      """The ordered candidate starts that apply to this problem (plan 029).
1627  
1628      0. "redundancy_off" / "strings_off" (Tier 18 redundancy): the same aircraft without redundancy, then (with
1629         battery strings) without the strings and their failure cases (plan 032).
1630      1. "electrical_off" (Tier 15 electrical layer with the thermal model): the same problem without the layer
1631         (plan 023: from the layer design without the thermal model IPOPT reaches local infeasibility). Without
1632         the thermal model it is a later fallback, after payload continuation.
1633         "thermal_off" (thermal model without the layer): the same problem without the thermal model (plan 028);
1634         from the generic guess, and with AeroBuildup from the thermal Scholz design, IPOPT can fail.
1635      2. "mass_objective" (objective "cost" only): the same 

- **L1625-1666 `default_starts`** builds the ordered list from the problem's switches. Each entry is there because one
  design-log entry found that problem hard from the generic guess:
  - L1646-1651 redundancy: `redundancy_off`, then `strings_off` (decision 032);
  - L1652-1653 thermal: `thermal_off` (decision 028), or `electrical_off` if the electrical layer is also on
    (decision 023: from the layer design without the thermal model IPOPT reaches local infeasibility);
  - L1654-1655 cost objective: `mass_objective`;
  - L1656-1657 AeroBuildup: `scholz_aero` (decision 025);
  - L1658-1661 ECM pack: `constant_battery` (decisions 022, 023), then `payload_continuation` for a fixed payload of at
    least 300 kg (decision 026);
  - L1662-1663 electrical layer without thermal: `electrical_off` as a late fallback;
  - L1664-1665 if any precursor start exists, the two perturbed starts; L1666 always `generic` last.
- **L1669-1689 `_precursor_problem`**: label → (requirements, assumptions, objective) of the simpler problem.
  `constant_battery` also switches the electrical layer off and uses the mass objective; `payload_continuation`
  lowers the payload to 85 %. Labels without a precursor (`caller`, `perturbed_*`, `generic`) return `None`.

In [51]:
target = HaloAssumptions()
starts = default_starts(req, target, "mass_takeoff")
print("default_starts(baseline):", starts)
check("baseline start order", starts == ("redundancy_off", "strings_off", "thermal_off", "scholz_aero", "constant_battery",
                                         "payload_continuation", "perturbed_low", "perturbed_high", "generic"))

def changed(problem_requirements, problem_assumptions):
    out = {f.name: getattr(problem_assumptions, f.name) for f in fields(HaloAssumptions)
           if getattr(problem_assumptions, f.name) != getattr(target, f.name)}
    out.update({f"req.{f.name}": getattr(problem_requirements, f.name) for f in fields(HaloRequirements)
                if getattr(problem_requirements, f.name) != getattr(req, f.name)})
    return out

for label in starts:
    problem = _precursor_problem(label, req, target, "mass_takeoff")
    if problem is None:
        print(f"  {label:<21} no precursor problem")
        continue
    nested = tuple(s for s in default_starts(*problem) if s not in ("perturbed_low", "perturbed_high")
                   + (("payload_continuation",) if label == "payload_continuation" else ()))
    print(f"  {label:<21} {problem[2]:<12} changes {changed(problem[0], problem[1])}")
    print(f"  {'':<21} its own starts: {nested}")

default_starts(baseline): ('redundancy_off', 'strings_off', 'thermal_off', 'scholz_aero', 'constant_battery', 'payload_continuation', 'perturbed_low', 'perturbed_high', 'generic')
PASS  baseline start order
  redundancy_off        mass_takeoff changes {'redundancy': False}
                        its own starts: ('thermal_off', 'scholz_aero', 'constant_battery', 'payload_continuation', 'generic')
  strings_off           mass_takeoff changes {'count_strings_battery': 1, 'failure_string_out': False}
                        its own starts: ('redundancy_off', 'thermal_off', 'scholz_aero', 'constant_battery', 'payload_continuation', 'generic')
  thermal_off           mass_takeoff changes {'thermal_model': False}
                        its own starts: ('redundancy_off', 'strings_off', 'scholz_aero', 'constant_battery', 'payload_continuation', 'generic')
  scholz_aero           mass_takeoff changes {'aerodynamics_model': 'scholz'}
                        its own starts: ('redundancy_off', 's

The baseline problem has every hard switch on, so its start list is the longest. The `redundancy_off` precursor is
itself the thermal + AeroBuildup + ECM problem without redundancy, and it is solved by the same strategy (its own
starts printed above), down to problems that converge from the generic guess. `select="first"` stops at the first
success, so in practice a cold baseline solve runs a chain of precursors once (they are cached, below) and then a few
target solves.

In [52]:
show_source("examples/halo_sizing.py", 1692, 1793)

# examples/halo_sizing.py  lines 1692-1793
1692  def perturbed_start(result, factor):
1693      """`result` with every design quantity, take-off mass and speeds scaled by `factor` (an initial guess only)."""
1694      design = HaloDesign(**{f.name: (getattr(result.design, f.name) * factor
1695                                      if getattr(result.design, f.name) is not None else None)
1696                             for f in fields(HaloDesign)})
1697      return replace(result, design=design, mass_takeoff_kg=result.mass_takeoff_kg * factor,
1698                     velocity_cruise_m_s=result.velocity_cruise_m_s * factor,
1699                     velocity_loiter_m_s=result.velocity_loiter_m_s * factor)
1700  
1701  
1702  def solve_halo_sizing_multistart(requirements=HaloRequirements(), assumptions=HaloAssumptions(), factors=None,
1703                                   verbose=False, max_iter=3000, objective="mass_takeoff", starts=None, select="first",
1704                            

- **L1692-1699 `perturbed_start`**: scales every design field, MTOM and both speeds by the factor. It is only an
  initial guess, so values outside a bound are fine (IPOPT pushes them inside): at the baseline, 0.85 x solidity is
  below its 0.06 bound and 1.15 x tip speed is above its Mach limit. Fixed numbers in the design (the turboshaft
  rating) are scaled too but ignored, because the solver reads them from the assumptions (L1116).
- **L1702-1724 `solve_halo_sizing_multistart`**: the public entry. `starts` defaults to `default_starts`; a caller's
  `initial` becomes the first start `"caller"` and the design that the perturbed starts scale.
- **L1727-1774 `_multistart`**, the loop over a **finite, ordered list**:
  - L1729-1733 validate labels;
  - L1738-1740 `caller`: the caller's design, recorded as the first precursor design;
  - L1741-1746 perturbed: scale the *first* precursor design found so far; skipped (recorded as failed) when no
    precursor exists yet;
  - L1747-1754 any other label: solve its precursor problem (L1749, cached) and use it as the start; a failed
    precursor is recorded and the loop moves on;
  - L1755-1764 one solve of the target from that start; a `RuntimeError` is recorded with IPOPT's last message line;
  - L1765-1766 `"first"` stops at the first success;
  - L1770-1774 with `"best"`, the lowest objective wins, ties within 1e-6 relative going to the earlier start; the
    returned result carries the `StartRecord`.
- **L1777-1790 `_precursor`**: solves a precursor by the same `_multistart`, with the default starts **minus** the
  perturbed starts and, inside a continuation precursor, minus payload continuation. This bounds the recursion: every
  precursor removes at least one switch (or lowers the payload once), so the tree is finite. The cache key
  (L1782) is the `repr` of requirements and assumptions, the objective, the identity of the factors object and the
  exclusions. A failed precursor is cached as `None`, so it is not retried.

In [53]:
low = perturbed_start(ref, factor_perturbation_low)
print(f"perturbed_low: MTOM {low.mass_takeoff_kg:.1f} kg, solidity {low.design.solidity:.4f} (bound 0.06), "
      f"tip speed x1.15 = {perturbed_start(ref, 1.15).design.speed_tip_m_s:.1f} m/s (bound 238.2)")
check("perturbed start scales design, MTOM and speeds by the factor",
      abs(low.mass_takeoff_kg - 0.85 * ref.mass_takeoff_kg) < 1e-9 and abs(low.design.area_wing_m2 - 0.85 * d.area_wing_m2) < 1e-12
      and abs(low.velocity_cruise_m_s - 0.85 * ref.velocity_cruise_m_s) < 1e-12)
record = StartRecord("scholz_aero", (StartAttempt("redundancy_off", False, message="x"),
                                     StartAttempt("scholz_aero", True, 6900.0, 6900.0),
                                     StartAttempt("generic", True, 6885.0, 6885.0)))
print("StartRecord successes:", [s.label for s in record.successes()], "spread", record.spread_mass_takeoff_kg(), "kg")
check("objective_value: payload is negated so lower is better",
      objective_value(replace(ref, mass_payload_kg=900.0), "payload") == -900.0)
try:
    solve_halo_sizing_multistart(starts=("nonsense",)); bad = False
except ValueError:
    bad = True
check("an unknown start label is rejected before any solve", bad)
print("cached baseline start record:", ref.start, "(None: the final integer solve was given `initial`, L957-958)")

perturbed_low: MTOM 5852.3 kg, solidity 0.0510 (bound 0.06), tip speed x1.15 = 273.9 m/s (bound 238.2)
PASS  perturbed start scales design, MTOM and speeds by the factor
StartRecord successes: ['scholz_aero', 'generic'] spread 15.0 kg
PASS  objective_value: payload is negated so lower is better
PASS  an unknown start label is rejected before any solve
cached baseline start record: None (None: the final integer solve was given `initial`, L957-958)


---
## 13. Named earlier baselines, max payload, bus-voltage enumeration

In [54]:
show_source("examples/halo_sizing.py", 1794, 1836)

# examples/halo_sizing.py  lines 1794-1836
1794  # Named earlier baselines, so each tier's notebook keeps reproducing its own result. `pre_plan035` pins the
1795  # settings plan 035 changed (drag corrections, redundancy, unit-built machines, gearbox stages) to their earlier
1796  # values for every set below.
1797  pre_plan035 = dict(drag_corrections=False, redundancy=False, machine_mass_model="torque_density", gearbox_stages=False)
1798  # Plans 037 and 038 (the drawn layout and the wing check): XV-15 fuselage calibration, turbogenerators at the tips,
1799  # round 12.8 m fuselage with the tail at 11.4 m, NDARC spar caps without a minimum gauge, XV-15 pylon ratio.
1800  pre_layout = dict(mass_factor_fuselage=None, turbogenerators_on_wing_tips=True, height_fuselage_m=None,
1801                    length_fuselage_m=42.1 * u.foot, x_horizontal_tail_m=11.4, ratio_radius_gyration_pylon=0.222,
1802                    ratio_depth_spar_cap=1.0, thickness_min_torque_box_m=0.0, offset_x_turbog

- **L1797 `pre_plan035`** pins the four settings decision 035 made default to their plan 030 values: no drag
  corrections, no redundancy, rubber torque-density machines, no gearbox stages. Every named set includes it.
- **L1800-1803 `pre_layout`** pins the plan 037/038 layout changes back: the XV-15 fuselage factor (`None`), engines
  at the tips, the round 42.1 ft fuselage with the tail at 11.4 m, NDARC spar caps (1.0) without minimum gauge, the
  XV-15 pylon ratio 0.222, and the plan 037 turbogenerator station (fields that matter only with the engines inside,
  kept for completeness).
- **L1804-1833** the requirement and assumption sets per tier. They are built with `**pre_plan035, **pre_layout`
  plus the tier's own switches; a set that differs only in one switch from a later one is written that way
  (`assumptions_tier11a = replace(assumptions_tier12, rotor_speed_physics=False)`, L1831).
- **L1834** the standard inverter blocking voltages for the bus-voltage options.

In [55]:
show_source("examples/halo_sizing.py", 1837, 1910)

# examples/halo_sizing.py  lines 1837-1910
1837  def solve_halo_max_payload(requirements=HaloRequirements(), assumptions=HaloAssumptions(), initial=None, **kwargs):
1838      """Maximum payload from `initial` (with the equivalent-circuit pack, a constant-battery design: plan 022).
1839  
1840      If that start fails and the pack is the equivalent circuit, the constant-battery maximum payload without the
1841      electrical layer (itself from `initial`) is tried as a second starting point. Starting points only: each attempt
1842      is one complete coupled solve.
1843      """
1844      try:
1845          return solve_halo_sizing(requirements, assumptions, objective="payload", initial=initial, **kwargs)
1846      except RuntimeError:
1847          if assumptions.battery_model != "ecm":
1848              raise
1849      start = solve_halo_sizing(requirements, replace(assumptions, battery_model="constant", electrical_layer=False),
1850                                objective="payload"

- **L1837-1851 `solve_halo_max_payload`**: the payload objective from `initial`; if that fails with the ECM pack, the
  constant-battery maximum-payload problem (without the electrical layer) is solved from `initial` and used as the
  start (decision 022: the ECM max-payload solve needs a constant-battery start). Two starting points, no loop.
- **L1854-1869 `assumptions_for_bus_voltage`**: a discrete bus-voltage option. Without DC/DC the pack sets the bus:
  series count = round(V / 3.6 V); with DC/DC the bus is regulated at V and the pack keeps its count. The inverter
  class is the smallest standard blocking voltage whose derated limit (x 0.75) covers the highest bus voltage
  (L1867-1868).
- **L1872-1910 `enumerate_bus_voltage`**: one independent sizing per option, an explicit enumeration of a discrete
  choice. Default start: the constant-battery aircraft without the layer (L1883-1884). Each option tries the common
  start, then the previous option's solution (L1889-1897); options that failed both are retried once from the
  nearest option that solved (L1900-1909). `None` marks an option that never solved.

In [56]:
print(f"{'V nominal':>9} {'series':>6} {'window V':>14} {'blocking V':>10} {'derated V':>9}")
for voltage_V in (540.0, 756.0, 800.0, 1000.0):
    option = assumptions_for_bus_voltage(HaloAssumptions(), voltage_V)
    window = bus_voltage_window(option)
    print(f"{voltage_V:9.0f} {option.count_series_battery:6d} {window.voltage_min_V:6.0f}-{window.voltage_max_V:6.0f} "
          f"{option.voltage_blocking_inverter_V:10.0f} {window.voltage_max_inverter_V:9.0f}")
check("800 V needs the 1,700 V class (932 V max bus > 0.75 x 1,200 V)",
      assumptions_for_bus_voltage(HaloAssumptions(), 800.0).voltage_blocking_inverter_V == 1700.0)
check("756 V (210 cells, 882 V max) fits the 1,200 V class",
      assumptions_for_bus_voltage(HaloAssumptions(), 756.0).voltage_blocking_inverter_V == 1200.0)
check("DC/DC at 800 V keeps 210 cells and the 1,200 V class",
      (lambda o: (o.count_series_battery, o.voltage_blocking_inverter_V))(assumptions_for_bus_voltage(HaloAssumptions(), 800.0, dcdc=True)) == (210, 1200.0))

V nominal series       window V blocking V derated V
      540    150    375-   630       1200       900
      756    210    525-   882       1200       900
      800    222    555-   932       1700      1275
     1000    278    695-  1168       1700      1275
PASS  800 V needs the 1,700 V class (932 V max bus > 0.75 x 1,200 V)
PASS  756 V (210 cells, 882 V max) fits the 1,200 V class
PASS  DC/DC at 800 V keeps 210 cells and the 1,200 V class


In [57]:
show_source("examples/halo_sizing.py", 1911, 1955)

# examples/halo_sizing.py  lines 1911-1955
1911  # Tier 20 (plan 024): the AFDD tiltrotor wing (the default from plan 026).
1912  assumptions_tier20 = HaloAssumptions(**pre_plan035, **pre_layout, thermal_model=False, wing_weight_model="afdd_tiltrotor",
1913                                       aerodynamics_model="simple")
1914  # Plan 027 reference: 900 kg, AFDD wing, AeroBuildup, no thermal model (13,639 lb).
1915  requirements_plan027 = HaloRequirements(mass_payload_kg=900.0)
1916  assumptions_plan027 = HaloAssumptions(**pre_plan035, **pre_layout, thermal_model=False)
1917  # Tier 18 (plan 032): the reference with the sensible redundancy set (2 lanes per rotor, 2 cross-strapped buses,
1918  # 2 battery strings; lane-out, bus-out and string-out hovers).
1919  assumptions_tier18 = HaloAssumptions(**dict(pre_plan035, redundancy=True), **pre_layout)
1920  # Plan 030 reference: 900 kg, AFDD wing, AeroBuildup, thermal model, rubber machines (14,037 lb).
1921  requirements_plan030 = HaloRe

- **L1911-1932** the remaining named sets: Tier 20, plans 027, 030, 037, 038 and Tier 18. They sit after
  `enumerate_bus_voltage`, apart from the others (L1804-1833), which is only an ordering oddity. `assumptions_plan037`
  is `pre_plan035` plus the four plan 038 settings reverted (it keeps the plan 037 layout); `assumptions_plan038` is
  `pre_plan035` alone.
- **L1935-1955 `__main__`**: one default solve and a printed summary (MTOM, geometry, ratings, binding margins, mass
  groups, segments, hovers). `python -m examples.halo_sizing` reproduces the v3.6 baseline in about 12 minutes.

**Named sets → aircraft versions** (`docs/AIRCRAFT_VERSIONS.md`; the take-off weights are the design log's, not
recomputed here):

| Set (line) | Built from | Version | Take-off weight |
|---|---|---|---|
| `requirements_tier10c` + `assumptions_tier11a` (L1804, L1831) | tier12 with the actuator disk; 250 kt, no hot day | v1.1 (v1.0 with the Geiss curve) | 18,506 lb |
| `requirements_tier10c` + `assumptions_tier12` (L1818-1822) | rubber engines, no recharge, no per-segment floor, power-sized machines | v1.2 | 17,228 lb |
| `requirements_tier12b` + `assumptions_tier12b` (L1805, L1824) | fixed engines, constant battery, Raymer wing, simple aero, power-sized machines | v2.0 | 14,877 lb |
| `requirements_tier12b` + `assumptions_tier16` (L1808) | torque-sized machines | v2.1 | 13,760 lb |
| `requirements_tier16` + `assumptions_tier16` (L1807-1808) | hot-day hover on | v3.0 | 13,760 lb |
| `assumptions_tier17` (L1828) | ECM pack (max payload) | v3.0.1 | not recorded |
| `requirements_plan022` + `assumptions_plan022` (L1812-1813) | ECM, 780 kg, Raymer wing | v3.0.2 | 14,436 lb |
| `requirements_plan022` + `assumptions_tier20` (L1912) | AFDD wing | v3.0.3 | 13,546 lb |
| `requirements_plan026` + `assumptions_plan026` (L1816-1817) | ECM + AFDD wing at 900 kg | v3.1 | 14,247 lb |
| `requirements_plan027` + `assumptions_plan027` (L1915-1916) | AeroBuildup | v3.2 (Scholz variant v3.2.1) | 13,639 lb |
| `assumptions_tier15` (L1833) on Scholz aero | electrical layer | v3.2.2 | 6,900 kg |
| `requirements_plan030` + `assumptions_plan030` (L1921-1922) | thermal model | v3.3 | 14,037 lb |
| `requirements_plan030` + `assumptions_tier18` (L1919) | redundancy | v3.3.1 | 14,342 lb |
| `requirements_plan037` + `assumptions_plan037` (L1925-1928) | drawn layout | v3.3.4 | 12,821 lb |
| `requirements_plan038` + `assumptions_plan038` (L1931-1932) | wing caps, pylon inertia, engine station | v3.3.5 | 13,038 lb |
| `HaloAssumptions(**pre_layout)` | plan 035 + 036 defaults with the old layout | v3.5 (no pinned set) | 16,303 lb |
| defaults (`solve_halo_sizing()`) | everything on | **v3.6** | **15,179 lb** |

In [58]:
versions_text = (repo_root / "docs" / "AIRCRAFT_VERSIONS.md").read_text()
named = sorted(n for n in dir(halo) if n.startswith("assumptions_") and n != "assumptions_for_bus_voltage")
for name in named:
    s = getattr(halo, name)
    diff = [f.name for f in fields(HaloAssumptions) if getattr(s, f.name) != getattr(HaloAssumptions(), f.name)]
    print(f"  {name:<22} {len(diff):2d} fields differ from the defaults; in AIRCRAFT_VERSIONS.md: {name in versions_text}")
check("every named assumption set is cited in docs/AIRCRAFT_VERSIONS.md", all(n in versions_text for n in named))
check("every named set pins the four plan 035 settings back",
      all(all(getattr(getattr(halo, n), k) == v for k, v in pre_plan035.items()) for n in named
          if n not in ("assumptions_tier18",)) and not assumptions_tier18.drag_corrections and assumptions_tier18.redundancy)
check("assumptions_plan038 is pre_plan035 alone; plan037 also reverts the four plan 038 settings",
      assumptions_plan038 == HaloAssumptions(**pre_plan035)
      and assumptions_plan037 == replace(assumptions_plan038, ratio_radius_gyration_pylon=0.222, ratio_depth_spar_cap=1.0,
                                         thickness_min_torque_box_m=0.0, offset_x_turbogenerators_m=1.0, z_turbogenerators_m=0.5))
same = [n for n in dir(halo) if n.startswith("requirements_") and getattr(halo, n) == HaloRequirements()]
print("requirement sets equal to today's defaults:", same)
docs_text = (repo_root / "docs" / "decisions" / "038-wing-caps-pylon.md").read_text()
check("decision 038 cites `assumptions_plan032`, which does not exist in the code",
      "assumptions_plan032" in docs_text and not hasattr(halo, "assumptions_plan032"))

  assumptions_plan022    17 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_plan026    16 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_plan027    15 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_plan030    14 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_plan037     9 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_plan038     4 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier11a    23 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True


  assumptions_tier12     22 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier12b    19 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True


  assumptions_tier15     16 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier16     18 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier17     17 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier18     13 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
  assumptions_tier20     16 fields differ from the defaults; in AIRCRAFT_VERSIONS.md: True
PASS  every named assumption set is cited in docs/AIRCRAFT_VERSIONS.md
PASS  every named set pins the four plan 035 settings back
PASS  assumptions_plan038 is pre_plan035 alone; plan037 also reverts the four plan 038 settings
requirement sets equal to today's defaults: ['requirements_plan026', 'requirements_plan027', 'requirements_plan030', 'requirements_plan037', 'requirements_plan038', 'requirements_tier16']
PASS  decision 038 cites `assumptions_plan032`, which does not exist in the code


---
## Findings

1. **Unused import** (`examples/halo_sizing.py` L86): `Xv15MassFactors` is imported and never used (checked above).
2. **`HaloSizingResult.machine_units` comment** (L961-962) says "(motor, generator) unit counts, relaxed (what the
   units would need) and fixed". After the integer re-solve (L1033) the first entry is the fixed count again, because
   L1002 evaluates the fixed `mass_model.count_units` set by `unit_machine` (L990-993). The baseline holds
   `{'motor': (2.0, 2), 'generator': (3.0, 3)}`; the relaxed counts of the first solve are lost.
3. **Duplicated SOC constraints.** The design margins `soc_end` (L1272) and `soc_after_engine_out_hover` (L1273) are
   the same inequalities that `build_mission` already writes (`mission.py` L126: `soc_end >= soc_floor` at every
   segment end, with `soc_floor` = the reserve for the mission, L1198, and the emergency floor for the reserve hover,
   L1220). Harmless while they are slack (margins +0.45 and +0.06 at the baseline), but if one became active the pair
   would be degenerate, the situation L1408-1409 explicitly avoids for the lane-out case.
4. **Hot-day SOC bookkeeping with the ECM pack** (L1245-1246): the SOC after the hot-day hover is chemical energy /
   pack energy (mean-OCV energy), while ECM points count coulombs (`battery_ecm.py` L274). The two differ by the
   ratio OCV(SOC) / mean OCV (computed below). It only affects the `soc_after_hot_day_hover` margin, which is slack.
5. **`solve_halo_sizing` docstring** (L1017-1021) lists the start order beginning with the thermal-off start; since
   plan 035 the redundancy starts come first (`default_starts` L1646-1651).
6. **Docs name a set that does not exist**: `docs/decisions/038-wing-caps-pylon.md` L27 and
   `docs/IMPLEMENTATION_NOTES.md` L1624 say `assumptions_plan032` reproduces 12,821 lb; the code (and
   `AIRCRAFT_VERSIONS.md`) call it `assumptions_plan037` (L1926).
7. **Cost lane count** (L1310, L1316), reported in C12: the cost block counts `count_rotors` motors, the topology has
   `count_rotors x count_lanes_motor`.
8. **Ordering**: six named sets (L1911-1932) are defined after `enumerate_bus_voltage`, apart from the others
   (L1794-1834). No functional effect.

In [59]:
pack = build_halo_battery(d, a)
soc_mid = (ref.soc_end + ref.soc_after_hover_hot) / 2
ratio = float(pack.voltage_open_circuit_V(soc_mid)) * float(pack.capacity_As) / float(pack.energy_capacity_J)
print(f"hot-day hover: SOC {ref.soc_end:.4f} -> {ref.soc_after_hover_hot:.4f}; energy-based / coulomb-based SOC drop = "
      f"OCV({soc_mid:.3f}) / mean OCV = {ratio:.4f}")
print(f"slack duplicated margins: soc_end {margins['soc_end'][0]:+.4f}, soc_after_engine_out_hover "
      f"{margins['soc_after_engine_out_hover'][0]:+.4f}")
check("the duplicated SOC margins are slack at the baseline (multiplier 0)",
      margins["soc_end"][1] < 1e-6 and margins["soc_after_engine_out_hover"][1] < 1e-6)

hot-day hover: SOC 0.4346 -> 0.3537; energy-based / coulomb-based SOC drop = OCV(0.394) / mean OCV = 0.9812
slack duplicated margins: soc_end +0.4485, soc_after_engine_out_hover +0.0589
PASS  the duplicated SOC margins are slack at the baseline (multiplier 0)


---
## Gotchas

- **Bounds are constraints, not IPOPT bounds.** `opti.variable(lower_bound=...)` adds a row `x / s >= lb / s` to `g`,
  so it has a multiplier in the scaled form: the price of a 1 % tighter bound is 1000 x μ x 0.01 x lb / s kg.
- **Equality duals have the opposite sign to inequality duals** for the same direction of tightening (toy cell in
  10.6). A negative closure dual is not "wrong": increasing A - B costs objective.
- **Degenerate pairs share one multiplier.** Turboshaft power and generator-gearbox power at the engine-out hover are
  the same function of the design; IPOPT splits the price between them arbitrarily. Add them.
- **`failure_lane_out=True` is not the same as "the lane-out case is in the problem"** (L1410-1412).
- **The hot-day hover sits at the end-of-mission mass and SOC**, the engine-out and failure hovers at MTOM and SOC
  0.30. They are different contingencies.
- **`initial` changes the code path.** `solve_halo_sizing(initial=r)` skips the multistart, and with strings on it
  uses the strings-off fallback; `solve_halo_sizing_once(initial=r)` is the plain single solve used in the x-ray.
- **The result's motor figures are per lane.** Two lane motors per rotor, four in all.
- **A perturbed start may lie outside the bounds**; that is legal for IPOPT, which projects it.

## Limits / what it can't do

- One coupled NLP with a local solver: every answer is a local optimum near the start. The multistart reduces the
  risk of a failed solve; `select="best"` measures the spread but does not guarantee the global optimum.
- Integer quantities (unit counts, stage count, lanes, strings, series cells) are handled by relax-and-round, a smooth
  staircase or enumeration, not by mixed-integer optimization.
- Failure states are symmetric (both rotors lose a lane); no asymmetric one-rotor states or roll trim (decision 032).
- Conversion flight is not part of the sizing problem; the corridor and trajectories are flown on the sized aircraft
  afterwards (C14).
- The whirl-flutter constraint is frequency placement, a proxy for a stability boundary.

## What's next (from `docs/NEXT_STEPS.md`)

1. Whirl flutter as a stability constraint inside the sizing: a rotor-pylon-wing eigenvalue model in CasADi, with
   the critical eigenpair as optimizer variables, keeping frequency placement as the simplest option.
2. Close the structural loop on the wing (buckling, cap-strain margin in the sizing).
3. Calibrate drag and rotor efficiency against flight data.

## Review questions

<details><summary>Why is MTOM a design variable with an equality, instead of being computed by iterating the mass buildup?</summary>

The parts depend on MTOM (structure through the design condition L1252, battery, machines and fuel through the points
flown at MTOM). Writing `m_TO / Σm(design, m_TO) == 1` (L1300) puts the fixed point inside the NLP: IPOPT converges
the closure and the optimality conditions together, and the closure's multiplier gives the mass growth factor
(2.54 kg/kg here) for free. An outer loop would hide the coupling from the optimizer and need its own convergence
test.
</details>

<details><summary>How do you turn the multiplier of a normalized margin into kilograms, and how did you validate it?</summary>

Each margin is `m >= 0` and the objective is MTOM/1000, so μ = d(MTOM/1000)/d(rhs) and requiring m >= 0.01 costs
1000 x μ x 0.01 kg. For the rotor-radius margin μ = 1.32, i.e. 13.2 kg per 1 %. Raising the clearance from 0.30 to
0.35 m shifts the margin by 0.05/4.832 = 1.03 %, predicting +13.7 kg; the warm re-solve gave +13.9 kg (1.5 %
second-order difference, same active set).
</details>

<details><summary>What sizes the Halo, in order?</summary>

The rotor-to-fuselage span limit (rotor wants more disk than the span allows), the end voltage of the engine-out hover
(sizes the parallel strings), turboshaft power at the engine-out and hot-day hovers and rotor/gearbox power at the
4,000 ft hover, hot-day heat rejection (exchanger), static margin and Cn_beta (tails), the bus-out lane torque. Of
the variables, only the tip speed (Mach 0.70) and solidity (0.06) are on bounds.
</details>

<details><summary>Why relax, round up and re-solve for the machine units, rather than round the relaxed design?</summary>

Rounding the relaxed counts up keeps torque and power capacity at least as large, so the integer design is feasible
for the machine limits. But the extra unit mass changes the closure, so every other variable must re-optimize: one
re-solve from the relaxed solution does that. Rounding alone would give a design that does not close. The 1e-3 offset
in `ceil(n - 1e-3)` keeps solver noise on an integer from adding a unit.
</details>

<details><summary>Why does the multistart not loop forever, and why are precursors cached?</summary>

The start list is finite and ordered; each precursor problem removes at least one hard switch (or lowers the payload
once) and is solved with the perturbed starts excluded and, inside a continuation, without further continuation, so
the recursion tree is finite. Precursors are cached by the repr of the problem, so the same simpler problem (for
example the non-redundant thermal problem) is solved once even when several starts need it, and a failure is cached
as `None`.
</details>

<details><summary>Why is the lane-out failure case missing from the baseline problem?</summary>

With 2 lanes on 2 buses, a bus out loses one lane per rotor and also routes power through the tie: it is the lane-out
state with more demand and heat. Its constraints dominate the lane-out ones, and including both would put duplicated
binding constraints in the problem (degenerate multipliers), so L1410-1412 skips the lane-out case.
</details>

In [60]:
check_summary()

69 of 69 checks passed
